---
# PCA MATERIALS ANALYSIS TOOL
---
### PURPOSE:
Performs Principal Component Analysis (PCA) on materials property descriptors by decomposing
the feature space into orthogonal principal components and mapping material categories into
a reduced latent descriptor space.
### ANALYSES PERFORMED:
1. Explained variance decomposition (eigenvector-based PCA)
2. PC loading biplot — descriptor contributions to PC1/PC2
3. PC1 vs PC2 scatter map — category clustering in latent space (static + interactive)
4. Debye temperature (Θ_D) statistics vs PC1/PC2 — binned mean ± std with 10–90th percentile band
5. Material count vs Θ_D distribution — histogram with smoothed envelope
6. Property contour maps — feature values projected onto PC1/PC2 space
7. Property threshold analysis — min/max property ranges at low/high Θ_D thresholds

**All figures exported as PDF; all data exported to a Excel workbook.**

### REQUIREMENTS:
- Python 3.12+ | numpy, pandas, matplotlib, scikit-learn, scipy, seaborn, openpyxl, plotly

### USAGE:
1. Run: pca_data_classifier.ipynb
2. Run: pca_interpretable_clusstering.ipynb
3. Follow prompts: data folder → output folder → symbol file → label confirmation → scatter options

In [21]:
# ── Required libraries ──────────────────────────────────────────────────────────

import time
import datetime
import os
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patches as patches
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.ticker import AutoMinorLocator, MaxNLocator
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from scipy.interpolate import griddata, make_interp_spline
from matplotlib.legend_handler import HandlerBase
from matplotlib.patches import Patch
import matplotlib.colors as mcolors
from matplotlib.collections import PolyCollection
try:
    import plotly.graph_objects as go
    import plotly.express as px
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False
    print("plotly not installed — interactive scatter disabled. Run: pip install plotly")
from joblib import Parallel, delayed, cpu_count as joblib_cpu_count
from concurrent.futures import ProcessPoolExecutor, as_completed 
import openpyxl
from openpyxl.styles import (Font, PatternFill, Alignment, Border, Side,
                              GradientFill)
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows
import seaborn as sns
try:
    import psutil
except ImportError:
    psutil = None
import platform
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['DejaVu Serif', 'Times New Roman', 'Palatino']
sns.set_style("ticks")

# ============================================================================
# KNOWN UNITS PER COLUMN KEY
# ============================================================================
COLUMN_UNITS: Dict[str, str] = {
    'NE': '', 'NA': '', 'BGAP': 'eV', 'EPA': 'eV/atom', 'FEPA': 'eV/atom',
    'EFMR': 'eV', 'TM': r'μ_B', 'MSITE': '', 'ASPIN': r'μ_B',
    'V': 'Å³', 'VPA': 'Å³/atom', 'D': 'g/cm³', 'PR': '', 'ANI': '',
    'BM': 'GPa', 'SM': 'GPa', 'YM': 'GPa', 'PUGR': '', 'IPUGR': '',
    'ENPA': 'eV/atom', 'vm': 'm/s', 'vl': 'm/s', 'vs': 'm/s',
    'DT_AGL': 'K', 'DT_A_AGL': 'K', 'DT': 'K',
    'Cp_300': 'J/(mol·K)', 'Cv_300': 'J/(mol·K)', 'TC_300': 'W/(m·K)',
    'TEX_300': '1/K', 'VIB_EN_300': 'eV/atom', 'VIB_FE_300': 'eV/atom',
    'gruneisen': '',
}


# ============================================================================
# TERMINAL COLOUR CODES AND WARNINGS
# ============================================================================
class Colors:
    HEADER = '\033[95m'
    BLUE   = '\033[94m'
    CYAN   = '\033[96m'
    GREEN  = '\033[92m'
    YELLOW = '\033[93m'
    RED    = '\033[91m'
    BOLD   = '\033[1m'
    ENDC   = '\033[0m'

    @classmethod
    def header(cls, text):
        print(f"\n{cls.HEADER}{cls.BOLD}{'='*80}\n{text.center(80)}\n{'='*80}{cls.ENDC}\n")

    @classmethod
    def block(cls, num, text):
        print(f"\n{cls.HEADER}{cls.BOLD}{'='*80}\nBLOCK {num}: {text}\n{'='*80}{cls.ENDC}\n")

    @classmethod
    def step(cls, num, text):
        print(f"\n{cls.BLUE}{cls.BOLD}[STEP {num}] {text}\n{'-'*80}{cls.ENDC}")

    @classmethod
    def success(cls, text): print(f"{cls.GREEN}  ✓  {text}{cls.ENDC}")
    @classmethod
    def warning(cls, text): print(f"{cls.YELLOW}  ⚠  {text}{cls.ENDC}")
    @classmethod
    def error(cls, text):   print(f"{cls.RED}  ✗  {text}{cls.ENDC}")
    @classmethod
    def info(cls, text):    print(f"{cls.CYAN}  ℹ  {text}{cls.ENDC}")


# ============================================================================
# FIGURES RELATED SETTINGS
# ============================================================================
class TwoColorLine:
    def __init__(self, color1, color2, linestyle='-', linewidth=2):
        self.color1 = color1; self.color2 = color2
        self.linestyle = linestyle; self.linewidth = linewidth


class HandlerTwoColorLine(HandlerBase):
    def __init__(self, gap_ratio=0.05, **kwargs):
        self.gap_ratio = gap_ratio
        super().__init__(**kwargs)

    def create_artists(self, legend, orig_handle, xdescent, ydescent,
                       width, height, fontsize, trans):
        gap  = width * self.gap_ratio
        half = (width - gap) / 2
        line1 = Line2D([xdescent, xdescent + half],
                       [ydescent + height / 2] * 2,
                       linestyle=orig_handle.linestyle,
                       linewidth=orig_handle.linewidth,
                       color=orig_handle.color1, transform=trans)
        line2 = Line2D([xdescent + half + gap, xdescent + width],
                       [ydescent + height / 2] * 2,
                       linestyle=orig_handle.linestyle,
                       linewidth=orig_handle.linewidth,
                       color=orig_handle.color2, transform=trans)
        return [line1, line2]

def _detect_environment() -> dict:
    # ── Jupyter detection ──────────────────────────────────────────────────
    try:
        import ipykernel
        in_jupyter = True
    except ImportError:
        in_jupyter = False

    # also catches VS Code notebooks, and JupyterLab
    try:
        shell = get_ipython().__class__.__name__
        if 'ZMQ' in shell or 'Kernel' in shell:
            in_jupyter = True
    except NameError:
        pass

    # ── SLURM / HPC detection ──────────────────────────────────────────────
    slurm_cpus = int(os.environ.get('SLURM_CPUS_PER_TASK', 0))
    in_slurm   = slurm_cpus > 0

    # ── CPU count ─────────────────────────────────────────────────────────
    # joblib_cpu_count() respects cgroup/SLURM limits, unlike os.cpu_count()
    machine_cpus = joblib_cpu_count()

    env = {
        'in_jupyter': in_jupyter,
        'in_slurm':   in_slurm,
        'slurm_cpus': slurm_cpus,
        'machine_cpus': machine_cpus,
    }

    # ── Report ─────────────────────────────────────────────────────────────
    Colors.info(f"Environment: {'Jupyter' if in_jupyter else 'Script'} | "
                f"{'SLURM (HPC)' if in_slurm else 'Local'} | "
                f"vCPUs available: {machine_cpus}")
    return env

def _select_backend(n_cpus: int, env: dict) -> dict:
    """
    Choose the right parallel backend based on the environment.

    Returns a config dict:
        backend  : 'joblib_threads' | 'joblib_loky' | 'processpool' | 'sequential'
        n_jobs   : int  (effective worker count)
        label    : str  (human-readable description for logging)
    """
    if n_cpus <= 1:
        return {'backend': 'sequential', 'n_jobs': 1,
                'label': 'Sequential (1 core)'}

    if env['in_jupyter']:
        # ProcessPoolExecutor is unsafe in Jupyter — always use joblib threads
        return {'backend': 'joblib_threads', 'n_jobs': n_cpus,
                'label': f'joblib threads ({n_cpus} cores) — Jupyter safe'}

    if env['in_slurm']:
        # HPC script — use ProcessPoolExecutor for true multiprocessing
        # cap at SLURM allocation to avoid over-subscribing the node
        effective = min(n_cpus, env['slurm_cpus'])
        return {'backend': 'processpool', 'n_jobs': effective,
                'label': f'ProcessPoolExecutor ({effective} cores) — HPC/SLURM'}

    # Local script (not Jupyter, not SLURM)
    # joblib loky is safer than raw multiprocessing and works on all OSes
    return {'backend': 'joblib_loky', 'n_jobs': n_cpus,
            'label': f'joblib loky ({n_cpus} cores) — local script'}


# ============================================================================
# UNIFIED PARALLEL RUNNER
# ============================================================================

def run_parallel(fn, args_list: list, backend_cfg: dict) -> list:
    backend = backend_cfg['backend']
    n_jobs  = backend_cfg['n_jobs']

    # ── Sequential ────────────────────────────────────────────────────────
    if backend == 'sequential' or len(args_list) <= 1:
        return [fn(args) for args in args_list]

    # ── joblib threads (Jupyter) ───────────────────────────────────────────
    if backend == 'joblib_threads':
        return Parallel(n_jobs=n_jobs, prefer='threads', verbose=0)(
            delayed(fn)(args) for args in args_list
        )

    # ── joblib loky (local script) ─────────────────────────────────────────
    if backend == 'joblib_loky':
        return Parallel(n_jobs=n_jobs, backend='loky', verbose=0)(
            delayed(fn)(args) for args in args_list
        )

    # ── ProcessPoolExecutor (HPC/SLURM) ───────────────────────────────────
    if backend == 'processpool':
        # Preserve order — map index → result
        results = [None] * len(args_list)
        indexed = list(enumerate(args_list))

        def _indexed_fn(idx_args):
            idx, args = idx_args
            return idx, fn(args)

        with ProcessPoolExecutor(max_workers=n_jobs) as executor:
            futures = {executor.submit(_indexed_fn, ia): ia[0]
                       for ia in indexed}
            for future in as_completed(futures):
                idx, result = future.result()
                results[idx] = result
        return results

    raise ValueError(f"Unknown backend: {backend}")
# ============================================================================
# SYMBOL CONVERTER
# ============================================================================
class SymbolConverter:
    def __init__(self, filepath: Optional[str] = None):
        self.mapping: Dict[str, str] = {}
        if filepath and os.path.exists(filepath):
            self._load_from_file(filepath)
        else:
            self._load_defaults()

    def _load_from_file(self, filepath: str) -> None:
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if not line or line.startswith('#'):
                        continue
                    if '=' in line and '->' not in line:
                        key, value = line.split('=', 1)
                    elif '->' in line:
                        key, value = line.split('->', 1)
                    else:
                        continue
                    key = key.strip(); value = value.strip()
                    if key and value:
                        self.mapping[key] = value
            Colors.success(f"Loaded {len(self.mapping)} symbol mappings from {filepath}")
        except Exception as e:
            Colors.warning(f"Failed to load symbol file: {e}. Using defaults.")
            self._load_defaults()

    def _load_defaults(self) -> None:
        self.mapping = {
            'SM': 'G_vrh', 'YM': 'E', 'VPA': 'V_atom',
            'D': 'rho', 'DT': 'Theta_D',
        }
        Colors.info("Using default symbol mappings")

    def get(self, key: str, default: Optional[str] = None) -> str:
        return self.mapping.get(key, default if default is not None else key)

    def get_axis_label(self, key: str) -> str:
        symbol = self.get(key)
        unit   = COLUMN_UNITS.get(key, '')
        return f"{symbol} [{unit}]" if unit else symbol

    def labels_for_columns(self, columns: List[str]) -> List[str]:
        return [self.get_axis_label(col) for col in columns]

    def report_coverage(self, columns: List[str]) -> None:
        mapped   = [c for c in columns if c in self.mapping]
        unmapped = [c for c in columns if c not in self.mapping]
        if mapped:   Colors.success(f"Symbol mappings found for: {mapped}")
        if unmapped: Colors.warning(f"No symbol mapping for: {unmapped} — using column name")


# ============================================================================
# CATEGORY CONFIGURATION
# ============================================================================
class CategoryConfig:
    FILENAME_MAP: Dict[str, str] = {
        "element": "Single Element",
        "metal":   "Metal & Metalloid Compounds",
        "boride":  "Borides",
        "carbide": "Carbides",
        "oxide":   "Oxides",
        "nitride": "Nitrides",
        "halide":  "Halides",
        "other":   "Other Compounds",
    }

    DISPLAY_ORDER: List[str] = [
        "Single Element", "Metal & Metalloid Compounds",
        "Nitrides", "Other Compounds", "Oxides",
        "Halides", "Borides", "Carbides",
    ]

    COLORS: Dict[str, str] = {
        "Single Element":              'teal',
        "Metal & Metalloid Compounds": 'tab:orange',
        "Oxides":                      '#FFFF33',
        "Nitrides":                    'fuchsia',
        "Halides":                     'lime',
        "Borides":                     'deepskyblue',
        "Carbides":                    'crimson',
        "Other Compounds":             'blue',
    }

    MARKERS: Dict[str, str] = {
        "Single Element":              'o',
        "Metal & Metalloid Compounds": 's',
        "Oxides":                      'P',
        "Nitrides":                    '*',
        "Halides":                     'D',
        "Borides":                     'X',
        "Carbides":                    '^',
        "Other Compounds":             'v',
    }

    MARKER_SIZES: Dict[str, int] = {
        'o': 95, 's': 65, 'D': 48, '^': 115,
        '*': 225, 'P': 115, 'X': 105, 'v': 75,
    }

    # Hex colours for Excel header fills (one per category)
    XL_HEADER_FILLS: Dict[str, str] = {
        "Single Element":              "008080",
        "Metal & Metalloid Compounds": "FFA500",
        "Oxides":                      "FFFF33",
        "Nitrides":                    "FF00FF",
        "Halides":                     "00FF00",
        "Borides":                     "00BFFF",
        "Carbides":                    "DC143C",
        "Other Compounds":             "0000FF",
    }

    @classmethod
    def categorize(cls, filename: str) -> str:
        name = os.path.basename(filename).lower()
        for keyword, label in cls.FILENAME_MAP.items():
            if keyword in name:
                return label
        return "Other Compounds"

class FileManager:

    @staticmethod
    def select_data_folder() -> str:
        Colors.step(1, "SELECT DATA FOLDER")
        while True:
            folder = input(
                f"\n{Colors.CYAN}  Enter full location of folder path containing classified Excel files: {Colors.ENDC}"
            ).strip().strip('"\'')
            if os.path.isdir(folder):
                Colors.success(f"Folder found: {folder}"); return folder
            Colors.error("Path not found — please try again.")

    @staticmethod
    def select_output_location() -> str:
        Colors.step(3, "SELECT OUTPUT LOCATION")
        while True:
            folder = input(
                f"\n{Colors.CYAN}  Enter folder path to save figures: {Colors.ENDC}"
            ).strip().strip('"\'')
            if folder:
                os.makedirs(folder, exist_ok=True)
                Colors.success(f"Output folder ready: {folder}"); break
            Colors.error("Path cannot be empty.")
        save_name = input(
            f"{Colors.CYAN}  Enter base filename for results (no extension): {Colors.ENDC}"
        ).strip()
        
        pca_folder = os.path.join(folder, "PCA")
        os.makedirs(pca_folder, exist_ok=True)
        Colors.info(f"PCA results will be saved in: {pca_folder}")
        return os.path.join(pca_folder, save_name)
        

    SYMBOL_FILE_NAME = "symbol_conversion.txt"

    @staticmethod
    def select_symbol_file() -> Optional[str]:
        Colors.step(4, "SYMBOL CONVERSION FILE")

        ans = input(
            f"\n{Colors.CYAN}  Load symbol_conversion.txt? (y/n): {Colors.ENDC}"
        ).strip().lower()

        if ans not in ("y", "yes"):
            Colors.info("Using default symbol mappings")
            return None

        while True:
            raw = input(
                f"{Colors.CYAN}  Enter file path or folder path: {Colors.ENDC}"
            ).strip().strip('"\'')

            base_path = os.path.abspath(os.path.expanduser(raw))

            # Case 1: direct file path provided
            if os.path.isfile(base_path):
                Colors.success(f"Symbol file found: {base_path}")
                return base_path

            # Case 2: user gave a directory → search inside it
            if os.path.isdir(base_path):
                candidate = os.path.join(base_path, FileManager.SYMBOL_FILE_NAME)
                if os.path.isfile(candidate):
                    Colors.success(f"Symbol file found (auto-detected): {candidate}")
                    return candidate

            Colors.error(
                f"File not found: '{base_path}' or inside folder — please try again."
            )

def _load_single(args):
    folder, file, required, categorize_fn = args
    full_path = os.path.join(folder, file)
    try:
        df = pd.read_excel(full_path, engine='openpyxl')
        extra_id_cols = [c for c in ['material_id', 'crystal_system'] if c in df.columns]
        available = extra_id_cols + [c for c in required if c in df.columns]
        missing   = [c for c in required if c not in df.columns]
        if missing:
            return None, file, f"missing columns {missing}"
        df = df[available].dropna()
        df["Category"] = categorize_fn(full_path)
        return df, file, None
    except Exception as e:
        return None, file, str(e)
# ============================================================================
# DATA LOADER
# ============================================================================
class DataLoader:
    REQUIRED_COLUMNS = ["formula", "SM", "YM", "VPA", "D", "DT"]
    FEATURE_COLUMNS  = ["SM", "YM", "VPA", "D", "DT"]

    @staticmethod
    def load_folder(folder: str,
                file_subset: Optional[List[str]] = None,
                n_cpus: int = 1,
                backend_cfg: dict = None) -> pd.DataFrame:

        Colors.step(6, "LOADING DATA")
        xlsx_files = sorted([f for f in os.listdir(folder) if f.endswith(".xlsx")])
        if not xlsx_files:
            Colors.error(f"No .xlsx files found in: {folder}"); sys.exit(1)

        if file_subset:
            xlsx_files = file_subset

        args_list = [
            (folder, f, DataLoader.REQUIRED_COLUMNS, CategoryConfig.categorize)
            for f in xlsx_files
        ]

        backend_cfg = backend_cfg or {'backend': 'sequential', 'n_jobs': 1,
                                       'label': 'Sequential'}
        Colors.info(f"Loading with: {backend_cfg['label']}")

        results = run_parallel(_load_single, args_list, backend_cfg)

        frames = []
        for df_loaded, fname, err in results:
            if err:
                Colors.warning(f"  {fname} — {err}, skipping.") if "missing" in str(err) \
                    else Colors.error(f"{fname}: {err}")
            else:
                Colors.success(
                    f"{fname}  →  '{df_loaded['Category'].iloc[0]}'  ({len(df_loaded):,} rows)"
                )
                frames.append(df_loaded)

        if not frames:
            Colors.error("No valid data loaded."); sys.exit(1)

        combined = pd.concat(frames, ignore_index=True)
        Colors.info(f"Total rows loaded: {len(combined):,}")
        return combined

    @staticmethod
    def detect_feature_columns(df: pd.DataFrame) -> List[str]:
        exclude = {'formula', 'Category'}
        pc_cols = {c for c in df.columns if c.startswith('PC')}
        return [c for c in df.columns
                if c not in exclude and c not in pc_cols and c in COLUMN_UNITS]

# ============================================================================
# PCA ALGORITHM
# ============================================================================
class PCARunner:
    def __init__(self, n_components: int = 5):
        self.n_components = n_components
        self.scaler = StandardScaler()
        self.pca    = PCA(n_components=n_components, svd_solver='randomized', random_state=42)

    def fit_transform(self, df: pd.DataFrame,
                      feature_cols: List[str]) -> Tuple[pd.DataFrame, np.ndarray]:
        Colors.block(1, "LATENT DESCRIPTOR SPACE CONSTRUCTION VIA EIGENVECTOR DECOMPOSITION")
        _t0 = time.perf_counter()
        X_scaled = self.scaler.fit_transform(df[feature_cols])
        print(f"  Scaler: {time.perf_counter()-_t0:.3f} s")
        _t0 = time.perf_counter()
        X_pca    = self.pca.fit_transform(X_scaled)
        print(f"  PCA decomposition: {time.perf_counter()-_t0:.3f} s")
        pc_cols  = [f'PC{i+1}' for i in range(X_pca.shape[1])]
        pca_df   = pd.DataFrame(X_pca, columns=pc_cols)
        result   = pd.concat([df.reset_index(drop=True), pca_df], axis=1)

        evr = self.pca.explained_variance_ratio_
        cum = np.cumsum(evr)
        print(f"\n  {'Component':<12} {'Variance':>10} {'Cumulative':>12}")
        print(f"  {'-'*36}")
        for i, (v, c) in enumerate(zip(evr, cum)):
            print(f"  PC{i+1:<9}  {v:>9.4f}   {c:>10.4f}")

        loadings = self.pca.components_.T
        Colors.success("PCA complete — Eigenvector decomposition successful")
        return result, loadings

    @property
    def explained_variance(self) -> np.ndarray:
        return self.pca.explained_variance_ratio_

# ============================================================================
#  DATA EXPORTER
# ============================================================================

_THIN_BORDER = Border(
    left=Side(style='thin'),  right=Side(style='thin'),
    top=Side(style='thin'),   bottom=Side(style='thin'),
)
_MED_BOTTOM = Border(bottom=Side(style='medium'))

def _xl_header_style(cell, hex_fill: str = "1F4E79",
                     font_hex: str = "FFFFFF", bold: bool = True,
                     font_size: int = 11) -> None:
    cell.font      = Font(bold=bold, color=font_hex, name='Arial',
                          size=font_size)
    cell.fill      = PatternFill("solid", fgColor=hex_fill)
    cell.alignment = Alignment(horizontal='center', vertical='center',
                               wrap_text=True)
    cell.border    = _THIN_BORDER

def _xl_data_style(cell, number_fmt: str = None,
                   align: str = 'center') -> None:
    cell.font      = Font(name='Arial', size=10)
    cell.alignment = Alignment(horizontal=align, vertical='center')
    cell.border    = _THIN_BORDER
    if number_fmt:
        cell.number_format = number_fmt

def _xl_auto_width(ws, min_w: int = 10, max_w: int = 40) -> None:
    for col_cells in ws.columns:
        length = max(
            (len(str(c.value)) if c.value is not None else 0)
            for c in col_cells
        )
        ws.column_dimensions[get_column_letter(col_cells[0].column)].width = \
            max(min_w, min(length + 3, max_w))

def _write_df_to_sheet(ws, df: pd.DataFrame,
                       header_fill: str = "1F4E79",
                       header_font: str = "FFFFFF",
                       num_fmt: str = '0.0000',
                       freeze: bool = True) -> None:
    
    # Header row
    for col_idx, col_name in enumerate(df.columns, start=1):
        cell = ws.cell(row=1, column=col_idx, value=str(col_name))
        _xl_header_style(cell, hex_fill=header_fill, font_hex=header_font)

    # Data rows
    for row_idx, row in enumerate(df.itertuples(index=False), start=2):
        for col_idx, value in enumerate(row, start=1):
            cell = ws.cell(row=row_idx, column=col_idx, value=value)
            fmt  = num_fmt if isinstance(value, (float, np.floating)) else None
            _xl_data_style(cell, number_fmt=fmt)

    if freeze:
        ws.freeze_panes = ws.cell(row=2, column=1)
    _xl_auto_width(ws)

# ---------- compute intermediate tables -------------------------------------

def _compute_pc_bin_stats(df: pd.DataFrame,
                           pc_col: str,
                           dt_col: str,
                           bin_size: float) -> pd.DataFrame:
    
    bins = np.arange(df[pc_col].min(), df[pc_col].max() + bin_size, bin_size)
    grp  = df.groupby(pd.cut(df[pc_col], bins=bins)).agg(
        mean_PC    = (pc_col, 'mean'),
        count      = (dt_col, 'size'),
        DT_mean    = (dt_col, 'mean'),
        DT_std     = (dt_col, 'std'),
        DT_min     = (dt_col, 'min'),
        DT_max     = (dt_col, 'max'),
        DT_p10     = (dt_col, lambda x: np.percentile(x, 10) if len(x) else np.nan),
        DT_p90     = (dt_col, lambda x: np.percentile(x, 90) if len(x) else np.nan),
    ).reset_index(drop=True)
    grp.rename(columns={'mean_PC': pc_col}, inplace=True)
    return grp

def _compute_dt_distribution(df: pd.DataFrame,
                              dt_col: str,
                              bin_size: float = 75) -> pd.DataFrame:
    bins = np.arange(df[dt_col].min(), df[dt_col].max() + bin_size, bin_size)
    df2  = df.copy()
    df2['bin_centre'] = pd.cut(df2[dt_col], bins=bins,
                                labels=bins[:-1]).astype(float)
    grp = df2.groupby('bin_centre').size().reset_index(name='count')
    grp.rename(columns={'bin_centre': f'{dt_col}_bin_centre (K)'}, inplace=True)
    return grp

def _compute_threshold_ranges(df: pd.DataFrame,
                               feature_cols: List[str],
                               dt_col: str,
                               low_cut: float = 350,
                               high_cut: float = 1000) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Return two DataFrames: property min/max for DT < low_cut and DT > high_cut.
    Rows = Threshold_K values; columns = feature min / max pairs.
    """
    # Low thresholds: iterate from dt_min to low_cut
    dt_min  = df[dt_col].min()
    low_thr = np.arange(max(-1000, dt_min), low_cut + 1, 2)

    # High thresholds: iterate from high_cut to dt_max
    dt_max   = df[dt_col].max()
    high_thr = np.arange(high_cut, dt_max + 1, 2)

    def build_range_df(thresholds, mask_fn):
        rows = []
        for thr in thresholds:
            sub = df[mask_fn(thr)]
            if len(sub) == 0:
                continue
            row = {'Threshold_K': thr, 'N_materials': len(sub)}
            for feat in feature_cols:
                if feat in sub.columns:
                    row[f'{feat}_min'] = sub[feat].min()
                    row[f'{feat}_max'] = sub[feat].max()
                    row[f'{feat}_mean'] = sub[feat].mean()
            rows.append(row)
        return pd.DataFrame(rows)

    df_low  = build_range_df(low_thr,  lambda t: df[dt_col] < t)
    df_high = build_range_df(high_thr, lambda t: df[dt_col] > t)
    return df_low, df_high

# ============================================================================
# USER OPTIONS
# ============================================================================
def ask_label_option() -> bool:
    Colors.step(5, "SCATTER PLOT OPTIONS")
    while True:
        ans = input(
            f"\n{Colors.CYAN}  Show formula labels on PC1 vs PC2 scatter plot? (y/n) [Default: n]: {Colors.ENDC}"
        ).strip().lower()
        if ans in ('y', 'yes'):   Colors.info("Labels ENABLED");  return True
        elif ans in ('n', 'no'): Colors.info("Labels DISABLED"); return False
        Colors.warning("Please enter y or n.")


def ask_feature_labels(feature_cols: List[str],
                       symbol_converter: SymbolConverter) -> List[str]:
    default = symbol_converter.labels_for_columns(feature_cols)
    # symbol_converter.report_coverage(feature_cols)
    # print(f"\n{Colors.BOLD}  Auto-generated feature labels:{Colors.ENDC}")
    # for key, lbl in zip(feature_cols, default):
    #     print(f"    {key}  →  {lbl}")
    # ans = input(f"{Colors.CYAN}  Use these labels? (y/n): {Colors.ENDC}").strip().lower()
    # if ans in ('y', 'yes'):
    #     return default
    # labels = []
    # for i, key in enumerate(feature_cols):
    #     lab = input(f"  Label for {key} (default '{default[i]}'): ").strip()
    #     labels.append(lab if lab else default[i])
    # return labels
    return default  # always use auto-generated labels; uncomment above to re-enable


# ============================================================================
# PLOT 1: Explained Variance Bar Chart  (BLOCK 1)
# ============================================================================
def plot_explained_variance(explained_variance: np.ndarray, save_base: str) -> None:
    mask   = explained_variance > 1e-3
    ev     = explained_variance[mask]
    n_bars = len(ev)

    gradient_cmap = LinearSegmentedColormap.from_list(
        '3d_blue', ['#0000E6', '#0343DF', '#0692E7'], N=2560)
    gradient_img = np.linspace(0, 1, 256).reshape(1, -1)
    gradient_img = np.repeat(gradient_img, 256, axis=0)

    fig, ax = plt.subplots(figsize=(6, 5))
    bar_width = 0.75

    for i, var in enumerate(ev):
        xc = i + 1
        xl, xr = xc - bar_width / 2, xc + bar_width / 2
        ax.imshow(gradient_img, aspect='auto',
                  extent=[xl, xr, 0, var], origin='lower', cmap=gradient_cmap,
                  vmin=0, vmax=1, zorder=2)
        rect = mpatches.FancyBboxPatch(
            (xl, 0), bar_width, var,
            boxstyle="square,pad=0", linewidth=1.82, edgecolor='darkblue',
            facecolor='none', zorder=3)
        ax.add_patch(rect)
        ax.text(xc, var + 0.01, f"{var:.2f}", ha='center', fontsize=16, zorder=4)

    ax.set_xlabel('PCA Principal Components', fontsize=20)
    ax.set_ylabel('Explained Variance', fontsize=20)
    ax.set_xticks(range(1, n_bars + 1))
    ax.tick_params(axis='both', which='both', direction='in',
                   labelsize=18.5, length=8, top=False, bottom=False,
                   left=True, right=True)
    ax.tick_params(axis='both', which='minor', direction='in',
                   length=3.5, width=1.0, top=False, bottom=False,
                   left=True, right=True)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.minorticks_on()
    ax.xaxis.set_minor_locator(AutoMinorLocator(10))
    ax.yaxis.set_minor_locator(AutoMinorLocator(10))
    ax.set_xlim(0.5, n_bars + 0.5)
    ax.set_ylim(0, max(ev) * 1.25)
    plt.tight_layout()
    out_png = f"{save_base}_Explained_Variance.png"
    out_pdf = f"{save_base}_Explained_Variance.pdf"
    plt.savefig(out_png, dpi=600, bbox_inches="tight")
    plt.savefig(out_pdf, dpi=600, bbox_inches="tight")
    plt.close()
    Colors.success(f"Variance plot saved: {out_png} | {out_pdf}")


# ============================================================================
# PLOT 2: PC Loading Biplot  (BLOCK 2)
# ============================================================================
def plot_loadings(loadings: np.ndarray,
                  feature_labels: List[str],
                  save_base: str) -> None:
    MARKERS = ['o', 's', 'D', '^', 'H']
    COLORS  = ['red', 'blue', 'green', 'orange', 'purple']

    fig, ax = plt.subplots(figsize=(8.5, 5.7))
    for i, label in enumerate(feature_labels):
        x, y = loadings[i, 0], loadings[i, 1]
        ax.scatter(x, y, marker=MARKERS[i % len(MARKERS)],
                   color=COLORS[i % len(COLORS)],
                   s=50, label=label, edgecolors='black', zorder=3)

    ax.axhline(0, color='black', linewidth=1, linestyle='--')
    ax.axvline(0, color='black', linewidth=1, linestyle='--')
    ax.add_patch(patches.Ellipse((0, 0), width=2, height=2,
                                  fill=False, color='black', linestyle='-', linewidth=1))
    ax.set_xlabel("PC1 Loading", fontsize=18)
    ax.set_ylabel("PC2 Loading", fontsize=18)
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1)
    ax.tick_params(axis='both', which='both', direction='in', labelsize=15, length=8,
                   top=True, bottom=True, left=True, right=True)
    ax.tick_params(axis='both', which='minor', direction='in', length=3.5, width=1.0,
                   top=True, bottom=True, left=True, right=True)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.minorticks_on()
    ax.xaxis.set_minor_locator(AutoMinorLocator(10))
    ax.yaxis.set_minor_locator(AutoMinorLocator(7))
    ax.legend(loc=[1.04, 0.42], fontsize=15, frameon=True, edgecolor='black')
    plt.tight_layout()
    out_png = f"{save_base}_PC_Loadings.png"
    out_pdf = f"{save_base}_PC_Loadings.pdf"
    plt.savefig(out_png, dpi=600, bbox_inches="tight")
    plt.savefig(out_pdf, dpi=600, bbox_inches="tight")
    plt.close()
    Colors.success(f"Loading plot saved: {out_png} | {out_pdf}")

# ============================================================================
# PLOT 3: PC1 vs PC2 Scatter  (BLOCK 2)
# ============================================================================
def plot_scatter(df: pd.DataFrame, save_base: str, show_labels: bool) -> None:
    fig, ax = plt.subplots(figsize=(18, 8))
    cfg = CategoryConfig

    for category in cfg.DISPLAY_ORDER:
        subset = df[df["Category"] == category]
        if subset.empty: continue
        marker = cfg.MARKERS[category]
        color  = cfg.COLORS[category]
        size   = cfg.MARKER_SIZES.get(marker, 65)
        ax.scatter(subset['PC1'], subset['PC2'],
                   color=color, marker=marker, edgecolor='black',
                   s=size, alpha=0.7, zorder=2)
        if show_labels:
            for _, row in subset.iterrows():
                ax.annotate(row['formula'], (row['PC1'], row['PC2']),
                            textcoords="offset points", xytext=(3, 3),
                            ha='left', fontsize=6, alpha=0.75)

    ax.axhline(0, color='black', linewidth=1.5, linestyle='--')
    ax.axvline(0, color='black', linewidth=1.5, linestyle='--')
    ax.set_xlabel('Principal Component 1', fontsize=25)
    ax.set_ylabel('Principal Component 2', fontsize=25)
    ax.tick_params(axis='both', which='major', direction='in', labelsize=22, length=16,
                   top=True, bottom=True, left=True, right=True)
    ax.tick_params(axis='both', which='minor', direction='in', length=7, width=1.0,
                   top=True, bottom=True, left=True, right=True)
    ax.yaxis.set_major_locator(MaxNLocator(5))
    ax.minorticks_on()
    ax.xaxis.set_minor_locator(AutoMinorLocator(20))
    ax.yaxis.set_minor_locator(AutoMinorLocator(10))

    legend_elements = [
        Line2D([0], [0], marker=cfg.MARKERS[cat], color='w', label=cat,
               markerfacecolor=cfg.COLORS[cat], markeredgecolor='black',
               markersize=15, linestyle='None')
        for cat in cfg.DISPLAY_ORDER if not df[df["Category"] == cat].empty
    ]
    ax.legend(handles=legend_elements, loc='best', fontsize=20,
              frameon=True, edgecolor='black')
    plt.tight_layout()
    out_png = f"{save_base}_PCA_Scatter.png"
    out_pdf = f"{save_base}_PCA_Scatter.pdf"
    plt.savefig(out_png, dpi=600, bbox_inches="tight")
    plt.savefig(out_pdf, dpi=600, bbox_inches="tight")
    plt.close()
    Colors.success(f"Scatter plot saved: {out_png} | {out_pdf}")

def plot_scatter_interactive(df: pd.DataFrame,
                             feature_cols: List[str],
                             symbol_converter: 'SymbolConverter',
                             save_base: str,
                             runner: 'PCARunner' = None,
                             inject_path: str = None):
    """
    Interactive Plotly scatter (PC1 vs PC2).
    - Plotly.js embedded (works offline)
    - HTML symbols in hover labels via symbol_converter (HTML-aware file)
    - Single export button with dropdown (PNG / SVG / JPEG / PDF)
    - Exports: white background, no buttons, no title, no subtitle
    """
    cfg             = CategoryConfig
    pc_cols_present = [c for c in df.columns if c.startswith('PC')]

    # ══════════════════════════════════════════════════════════════════════
    # 1. Label builder 
    # ══════════════════════════════════════════════════════════════════════
    def fmt_label(feat: str) -> str:
        symbol = symbol_converter.get(feat, feat)
        unit   = COLUMN_UNITS.get(feat, '')
        return f"{symbol} ({unit})" if unit else symbol

    print("\n" + "═" * 62)
    print("  Building interactive plot ...")
    print("═" * 62 + "\n")

    # ══════════════════════════════════════════════════════════════════════
    # 2. Colour map — matplotlib names → hex
    # ══════════════════════════════════════════════════════════════════════
    MPL_TO_HEX: Dict[str, str] = {
        'tab:blue':    '#1f77b4', 'tab:orange': '#ff7f0e',
        'tab:green':   '#2ca02c', 'tab:red':    '#d62728',
        'tab:purple':  '#9467bd', 'tab:brown':  '#8c564b',
        'tab:pink':    '#e377c2', 'tab:gray':   '#7f7f7f',
        'tab:grey':    '#7f7f7f', 'tab:olive':  '#bcbd22',
        'tab:cyan':    '#17becf', 'teal':       '#008080',
        'fuchsia':     '#ff00ff', 'lime':       '#00ff00',
        'deepskyblue': '#00bfff', 'crimson':    '#dc143c',
        'blue':        '#0000ff', 'red':        '#ff0000',
    }
    def safe_color(c: str) -> str:
        return MPL_TO_HEX.get(c, c)

    PLOTLY_MARKERS = {
        'o': 'circle',       's': 'square',
        'P': 'cross',        '*': 'star',
        'D': 'diamond',      'X': 'x',
        '^': 'triangle-up',  'v': 'triangle-down',
    }

    # ══════════════════════════════════════════════════════════════════════
    # 3. Build one trace per category
    # ══════════════════════════════════════════════════════════════════════
    fig = go.Figure()

    for category in cfg.DISPLAY_ORDER:
        subset = df[df["Category"] == category].copy()
        if subset.empty:
            continue

        marker_symbol = cfg.MARKERS.get(category, 'o')
        color         = safe_color(cfg.COLORS.get(category, 'tab:blue'))
        plotly_marker = PLOTLY_MARKERS.get(marker_symbol, 'circle')
        mpl_size      = cfg.MARKER_SIZES.get(marker_symbol, 65)
        plotly_size   = max(9, min(10, int(mpl_size ** 0.52)))

        hover_texts = []
        for _, row in subset.iterrows():
            lines = []

            formula = row.get('formula', '')
            crystal = row.get('crystal_system', '')
            if formula:
                display = f"<b>{formula} ({crystal})</b>" if crystal else f"<b>{formula}</b>"
                lines.append(display)
            lines.append(f"<i>{category}</i>")
            lines.append("─" * 26)

            lines.append("<b>PCA Scores</b>")
            for pc in pc_cols_present:
                if pc in row:
                    lines.append(f"&nbsp;&nbsp;{pc}: {row[pc]:+.4f}")

            lines.append("─" * 26)
            lines.append("<b>Properties</b>")
            for feat in feature_cols:
                if feat not in row:
                    continue
                lbl     = fmt_label(feat)
                val     = row[feat]
                val_str = f"{val:.4f}" if isinstance(val, float) else str(val)
                lines.append(f"&nbsp;&nbsp;{lbl}: {val_str}")

            hover_texts.append("<br>".join(lines))

        fig.add_trace(go.Scatter(
            x=subset['PC1'],
            y=subset['PC2'],
            mode='markers',
            name=category,
            text=hover_texts,
            hovertemplate="%{text}<extra></extra>",
            marker=dict(
                symbol=plotly_marker,
                size=plotly_size,
                color=color,
                line=dict(color='black', width=0.9),
                opacity=0.85,
            ),
        ))

    # ══════════════════════════════════════════════════════════════════════
    # 4. Layout
    # ══════════════════════════════════════════════════════════════════════
    fig.update_layout(
        title=dict(
            text=(
                "<b>PCA Scatter — PC1 vs PC2</b><br>"
                "<sup style='color:#333'>&nbsp;&nbsp;"
                "Click legend = hide/show  |  "
                "Double-click legend = isolate  |  "
                "Scroll = zoom  |  Drag = pan  |  "
                "Box / Lasso (toolbar) = select region"
                "</sup>"
            ),
            x=0.5, xanchor='center',
        ),
        xaxis=dict(
            title=dict(text="Principal Component 1"),
            title_standoff=14,
            zeroline=True,  zerolinecolor='black',  zerolinewidth=0.7,
            showgrid=True,  mirror=True,  ticks='inside',
            showline=True,  showspikes=False,
        ),
        yaxis=dict(
            title=dict(text="Principal Component 2"),
            title_standoff=14,
            zeroline=True,  zerolinecolor='black',  zerolinewidth=0.7,
            showgrid=True,  mirror=True,  ticks='inside',
            showline=True,  showspikes=False,
        ),
        legend=dict(
            title=dict(text="<b>Category</b>"),
            bgcolor='rgba(255,255,255,0.95)',
            x=1.02, y=1.0,
            xanchor='left', yanchor='top',
            itemclick='toggle',
            itemdoubleclick='toggleothers',
        ),
        plot_bgcolor='white',
        paper_bgcolor='#ffffff',
        hovermode='closest',
        hoverlabel=dict(bgcolor='white', namelength=-1),
        width=1500,
        height=680,
        margin=dict(l=80, r=230, t=120, b=90),
    )

    # ── Dashed zero-line shapes ───────────────────────────────────────────
    for shape_kw in [
        dict(x0=0, x1=0,
             y0=df['PC2'].min() - 1, y1=df['PC2'].max() + 1),
        dict(x0=df['PC1'].min() - 1, x1=df['PC1'].max() + 1,
             y0=0, y1=0),
    ]:
        fig.add_shape(type='line', **shape_kw,
                      line=dict(color='rgba(0,0,0,0.3)', width=1.2, dash='dash'),
                      layer='below')

    # ══════════════════════════════════════════════════════════════════════
    # 5. Write HTML, then inject export UI
    # ══════════════════════════════════════════════════════════════════════
    out        = f"{save_base}_PCA_Scatter_Interactive.html"
    base_fname = os.path.basename(save_base) + '_PCA_scatter'

    config = dict(
        displayModeBar=True,
        displaylogo=False,
        scrollZoom=True,
        modeBarButtonsToAdd=[
            'drawline', 'drawopenpath', 'drawclosedpath',
            'drawcircle', 'drawrect', 'eraseshape',
        ],
        toImageButtonOptions=dict(
            format='png',
            filename=base_fname,
            height=680, width=1500, scale=3,
        ),
    )

    fig.write_html(
        out,
        include_plotlyjs=True,
        config=config,
        full_html=True,
    )

    if inject_path is None or not os.path.isfile(inject_path):
        Colors.warning("Inject file not found — interactive plot saved without custom UI.")
        INJECT = ""
    else:
        with open(inject_path, 'r', encoding='utf-8') as fh:
            INJECT = fh.read().replace('{base_fname}', base_fname)

    with open(out, 'r', encoding='utf-8') as fh:
        html = fh.read()

    html = html.replace('</body>', INJECT + '\n</body>')

    with open(out, 'w', encoding='utf-8') as fh:
        fh.write(html)

    Colors.success(f"Interactive scatter saved: {out}")
    
# ============================================================================
# PLOT 4: Material Count vs PC1 / PC2  (BLOCK 3)
# ============================================================================

def plot_material_count_vs_components(df: pd.DataFrame, save_base: str) -> None:
    Colors.info("Generating Material Count vs PC1/PC2 plot...")

    sorted_data  = df.sort_values("PC1").reset_index(drop=True)
    bin_size_PC1 = 0.950
    bin_size_PC2 = 0.475

    bins_PC1 = np.arange(sorted_data["PC1"].min(),
                          sorted_data["PC1"].max() + bin_size_PC1, bin_size_PC1)
    bins_PC2 = np.arange(sorted_data["PC2"].min(),
                          sorted_data["PC2"].max() + bin_size_PC2, bin_size_PC2)

    # ── Aggregation ───────────────────────────────────────────────────────
    def _bin_count(data: pd.DataFrame, pc_col: str,
                   bins: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        grp = (
            data.groupby(pd.cut(data[pc_col], bins=bins))
            .agg(mean_pc=(pc_col, "mean"), count=("DT", "size"))
            .reset_index(drop=True)
        )
        x = grp["mean_pc"].values
        y = np.where(grp["count"].values == 0, np.nan,
                     grp["count"].values.astype(float))
        return x, y

    x_PC1, y_PC1 = _bin_count(sorted_data, "PC1", bins_PC1)
    x_PC2, y_PC2 = _bin_count(sorted_data, "PC2", bins_PC2)

    # ── Smooth-segment builder ────────────────────────────────────────────
    def _smooth_segments(x: np.ndarray, y: np.ndarray,
                         n_pts: int = 400) -> Tuple[list, list]:
        segs_x, segs_y = [], []
        valid   = ~np.isnan(y)
        indices = np.where(np.diff(valid.astype(int)))[0] + 1
        splits  = np.split(np.arange(len(x)), indices)
        for idx_run in splits:
            if len(idx_run) < 2:
                continue
            xs, ys = x[idx_run], y[idx_run]
            if np.any(np.isnan(ys)):
                continue
            k    = min(3, len(xs) - 1)
            xnew = np.linspace(xs.min(), xs.max(), n_pts)
            ynew = make_interp_spline(xs, ys, k=k)(xnew)
            segs_x.append(xnew)
            segs_y.append(ynew)
        return segs_x, segs_y

    segs_x_PC1, segs_y_PC1 = _smooth_segments(x_PC1, y_PC1)
    segs_x_PC2, segs_y_PC2 = _smooth_segments(x_PC2, y_PC2)

    def _marker_idx(x: np.ndarray, n_markers: int = 10) -> np.ndarray:
        n = max(1, min(n_markers, len(x)))
        return np.linspace(0, len(x) - 1, n, dtype=int)

    fig, ax1 = plt.subplots(figsize=(7.5, 5))

    # ── PC1 — bottom x-axis (dark blue, triangles) ───────────────────────
    label_added = False
    for xs, ys in zip(segs_x_PC1, segs_y_PC1):
        ax1.plot(xs, ys, "-", color="darkblue", lw=1.8, zorder=3)
        mi = _marker_idx(xs)
        ax1.plot(xs[mi], ys[mi], "^", color="darkblue", markersize=6,
                 label="PC1" if not label_added else "_nolegend_", zorder=4)
        label_added = True
    ax1.set_xlabel("PC1", fontsize=21, color="black")
    ax1.set_ylabel("Number of Materials", fontsize=21)
    ax1.tick_params(axis="x", colors="black", labelsize=18.5,
                    direction="in", length=8, width=1.5)
    ax1.tick_params(axis="y", labelsize=18.5,
                    direction="in", length=8, width=1.5,
                    right=True, left=True)
    ax1.tick_params(axis="both", which="minor",
                    direction="in", length=4, width=1.5,
                    right=True, left=True)
    ax1.minorticks_on()
    ax1.yaxis.set_minor_locator(AutoMinorLocator(7))
    ax1.xaxis.set_minor_locator(AutoMinorLocator(15))
    all_y = np.concatenate(segs_y_PC1 + segs_y_PC2) if segs_y_PC2 else np.concatenate(segs_y_PC1)
    y_hi  = np.ceil(np.nanmax(all_y) / 50) * 50
    step  = max(50, round(y_hi / 5 / 50) * 50)
    ax1.set_ylim(0, y_hi * 1.08)
    ax1.set_yticks(np.arange(0, y_hi + 1, step))

    # ── PC2 — top x-axis (red, diamonds) ─────────────────────────────
    ax2 = ax1.twiny()
    label_added = False
    for xs, ys in zip(segs_x_PC2, segs_y_PC2):
        ax2.plot(xs, ys, "-", color="crimson", lw=1.8, zorder=3)
        mi = _marker_idx(xs)
        ax2.plot(xs[mi], ys[mi], "D", color="crimson", markersize=5,
                 label="PC2" if not label_added else "_nolegend_", zorder=4)
        label_added = True
    ax2.set_xlabel("PC2", fontsize=21, color="black")
    ax2.tick_params(axis="x", colors="black", labelsize=18.5,
                    direction="in", length=8, width=1.5)
    ax2.tick_params(axis="both", colors="black", which="minor",
                    direction="in", length=4, width=1.5)
    ax2.minorticks_on()
    ax2.xaxis.set_minor_locator(AutoMinorLocator(15))
    ax2.yaxis.set_minor_locator(AutoMinorLocator(7))
    _leg_pc1 = Line2D([0], [0], marker='^', color='darkblue', lw=1.8,
                      markersize=7, linestyle='-',
                      label='PC1: Number of Materials')
    _leg_pc2 = Line2D([0], [0], marker='D', color='crimson', lw=1.8,
                      markersize=6, linestyle='-',
                      label='PC2: Number of Materials')
    ax1.legend(handles=[_leg_pc1, _leg_pc2],
               fontsize=14, fancybox=True, edgecolor='k',
               loc='upper right', framealpha=1, ncol=1)
    plt.tight_layout()
    out_png = f"{save_base}_NumMaterials_vs_PC1_PC2.png"
    out_pdf = f"{save_base}_NumMaterials_vs_PC1_PC2.pdf"
    plt.savefig(out_png, dpi=600, bbox_inches="tight")
    plt.savefig(out_pdf, dpi=600, bbox_inches="tight")
    plt.close()
    Colors.success(f"Material count plot saved: {out_png}  |  {out_pdf}")

# ============================================================================
# PLOT 5: Θ_D Statistics vs PC1/PC2  (BLOCK 3)
# ============================================================================
def plot_theta_d_statistics_vs_components(df: pd.DataFrame,
                                          save_base: str,
                                          symbol_converter: SymbolConverter,
                                          dt_col: str = 'DT') -> None:
    Colors.info(f"Generating {dt_col} statistics vs PC1/PC2 plot...")
    y_label = symbol_converter.get_axis_label(dt_col)

    sorted_data  = df.sort_values("PC1").reset_index(drop=True)
    bin_size_PC1 = 0.950
    bin_size_PC2 = 0.475
    bins_PC1 = np.arange(sorted_data["PC1"].min(),
                          sorted_data["PC1"].max() + bin_size_PC1, bin_size_PC1)
    bins_PC2 = np.arange(sorted_data["PC2"].min(),
                          sorted_data["PC2"].max() + bin_size_PC2, bin_size_PC2)

    def agg_stats(grp_col, mean_col, bins):
        return sorted_data.groupby(pd.cut(sorted_data[grp_col], bins=bins)).agg(
            **{mean_col:       (grp_col, "mean"),
               "mean_theta":  (dt_col, "mean"),
               "std_theta":   (dt_col, "std"),
               "q10_theta":   (dt_col, lambda x: np.percentile(x, 10) if len(x) > 0 else np.nan),
               "q90_theta":   (dt_col, lambda x: np.percentile(x, 90) if len(x) > 0 else np.nan),
               "count":       (dt_col, "size")}
        ).reset_index()

    grouped_PC1 = agg_stats("PC1", "mean_PC1", bins_PC1)
    grouped_PC2 = agg_stats("PC2", "mean_PC2", bins_PC2)

    fig, ax1 = plt.subplots(figsize=(7.5, 5))
    ax1.errorbar(grouped_PC1["mean_PC1"], grouped_PC1["mean_theta"],
                 yerr=grouped_PC1["std_theta"], fmt='^-', color="darkblue",
                 ecolor="darkblue", elinewidth=1.5, capsize=6, zorder=10, label=r"PC1: $\Theta_D$ Mean $\pm$ Std")
    ax1.fill_between(grouped_PC1["mean_PC1"],
                     grouped_PC1["q10_theta"], grouped_PC1["q90_theta"],
                     color="blue", alpha=0.2, zorder=5)

    ax1.set_xlabel("PC1", fontsize=21, color="black")
    ax1.set_ylabel(y_label, fontsize=21)
    ax1.tick_params(axis="x", colors="black", labelsize=18.5, direction='in', length=8, width=1.5)
    ax1.tick_params(axis="y", labelsize=18.5, direction='in', length=8, right=True, left=True, width=1.5)
    ax1.tick_params(axis='both', which='minor', direction='in', length=4, right=True, left=True, width=1.5)
    ax1.minorticks_on()
    ax1.yaxis.set_minor_locator(AutoMinorLocator(7))
    ax1.xaxis.set_minor_locator(AutoMinorLocator(15))

    ax2 = ax1.twiny()
    ax2.errorbar(grouped_PC2["mean_PC2"], grouped_PC2["mean_theta"],
                 yerr=grouped_PC2["std_theta"], fmt='D-', color="crimson",
                 ecolor="crimson", elinewidth=1.5, capsize=3, zorder=10, label=r"PC2: $\Theta_D$ Mean $\pm$ Std")
    ax2.fill_between(grouped_PC2["mean_PC2"],
                     grouped_PC2["q10_theta"], grouped_PC2["q90_theta"],
                     color="red", alpha=0.2, zorder=5)

    ax2.set_xlabel("PC2", fontsize=21, color="black")
    ax2.tick_params(axis="x", colors="black", labelsize=18.5, direction='in', length=8, width=1.5)
    ax2.tick_params(axis='both', colors="black", which='minor', direction='in', length=4, width=1.5)
    ax2.minorticks_on()
    ax2.xaxis.set_minor_locator(AutoMinorLocator(15))
    ax2.yaxis.set_minor_locator(AutoMinorLocator(7))
    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    ax1.set_ylim(0, 3000)
    ax1.legend(handles1 + handles2, labels1 + labels2,
               fontsize=14, fancybox=True, edgecolor="k", loc="upper center", framealpha=1, ncol=2)

    plt.tight_layout()
    out_png = f"{save_base}_ThetaD_Stats_vs_PC1_PC2.png"
    out_pdf = f"{save_base}_ThetaD_Stats_vs_PC1_PC2.pdf"
    plt.savefig(out_png, dpi=600, bbox_inches="tight")
    plt.savefig(out_pdf, dpi=600, bbox_inches="tight")
    plt.close()
    Colors.success(f"Θ_D statistics plot saved: {out_png}  |  {out_pdf}")

# ============================================================================
# PLOT 6: Material Count vs Θ_D  (BLOCK 3)
# ============================================================================
def plot_theta_d_distribution(df: pd.DataFrame,
                               save_base: str,
                               symbol_converter: SymbolConverter,
                               dt_col: str = 'DT') -> None:
    Colors.info(f"Generating {dt_col} distribution plot...")
    x_label = symbol_converter.get_axis_label(dt_col)

    sorted_data = df.sort_values(dt_col).reset_index(drop=True)
    bin_size    = 75
    bins        = np.arange(sorted_data[dt_col].min(),
                             sorted_data[dt_col].max() + bin_size, bin_size)

    sorted_data["bin"] = pd.cut(sorted_data[dt_col], bins=bins, labels=bins[:-1])
    grouped_total = sorted_data.groupby("bin").size().reset_index(name="count")
    grouped_total["mean_theta"] = grouped_total["bin"].astype(float)
    grouped_total["count"]      = grouped_total["count"].replace(0, np.nan)

    def gradient_bar(ax, x, height, width=60,
                     color_top="#B199FF", color_bottom="#6F00FF"):
        cmap   = LinearSegmentedColormap.from_list("bar_cmap", [color_bottom, color_top])
        grad   = np.linspace(0, 1, 250)
        grad2d = np.outer(grad, grad)
        ax.imshow(grad2d, extent=(x - width/2, x + width/2, 0, height),
                  aspect="auto", origin="lower", cmap=cmap,
                  zorder=1, alpha=1, interpolation="bicubic")
        ax.plot([x - width/2, x + width/2, x + width/2, x - width/2, x - width/2],
                [0, 0, height, height, 0], color="#4B0082", lw=1.3, zorder=2)

    def smooth_segments(x, y, points=500):
        segs_x, segs_y = [], []
        is_valid = ~np.isnan(y)
        start = None
        for i, valid in enumerate(is_valid):
            if valid and start is None:
                start = i
            elif not valid and start is not None:
                if i - start > 1:
                    xs, ys = x[start:i], y[start:i]
                    xnew   = np.linspace(xs.min(), xs.max(), points)
                    ynew   = make_interp_spline(xs, ys, k=3)(xnew)
                    segs_x.append(xnew); segs_y.append(ynew)
                start = None
        if start is not None and len(x) - start > 1:
            xs, ys = x[start:], y[start:]
            xnew   = np.linspace(xs.min(), xs.max(), points)
            ynew   = make_interp_spline(xs, ys, k=3)(xnew)
            segs_x.append(xnew); segs_y.append(ynew)
        return segs_x, segs_y

    fig, ax = plt.subplots(figsize=(7, 4.5))
    for _, row in grouped_total.iterrows():
        if not np.isnan(row["count"]):
            gradient_bar(ax, row["mean_theta"], row["count"], width=55,
                         color_top="#B199FF", color_bottom="#6F00FF")

    x = grouped_total["mean_theta"].values
    y = grouped_total["count"].values
    segs_x, segs_y = smooth_segments(x, y)
    for xs, ys in zip(segs_x, segs_y):
        ax.plot(xs, ys, color="orangered", linewidth=3.5, linestyle='-', zorder=3)

    ax.set_xlabel(x_label, fontsize=18)
    ax.set_ylabel("Number of Materials", fontsize=18)
    ax.tick_params(axis='both', labelsize=16)
    ax.tick_params(axis='both', which='both', direction='in', length=8, width=1.5,
                   top=True, bottom=True, left=True, right=True)
    ax.tick_params(axis='both', which='minor', direction='in', length=3.5, width=1.5,
                   top=True, bottom=True, left=True, right=True)
    ax.minorticks_on()
    ax.xaxis.set_minor_locator(AutoMinorLocator(15))
    ax.yaxis.set_minor_locator(AutoMinorLocator(7))
    plt.xlim(-100, 2300)
    plt.tight_layout()
    out_png = f"{save_base}_Count_vs_ThetaD.png"
    out_pdf = f"{save_base}_Count_vs_ThetaD.pdf"
    plt.savefig(out_png, dpi=600, bbox_inches="tight")
    plt.savefig(out_pdf, dpi=600, bbox_inches="tight")
    plt.close()
    Colors.success(f"Θ_D distribution plot saved: {out_png}  |  {out_pdf}")

# ============================================================================
# PLOT 7: PCA Contour Maps  (BLOCK 3)
# ============================================================================
def _build_contour_grid(x, y, z, resolution=150):
    xi = np.linspace(x.min(), x.max(), resolution)
    yi = np.linspace(y.min(), y.max(), resolution)
    xi, yi = np.meshgrid(xi, yi)
    zi = griddata((x, y), z, (xi, yi), method='linear')
    return xi, yi, np.clip(zi, np.nanmin(z), np.nanmax(z))

def _draw_single_contour(ax, xi, yi, zi, label, cmap='jet'):
    levels  = np.linspace(np.nanmin(zi), np.nanmax(zi), 1000)
    contour = ax.contourf(xi, yi, zi, levels=levels, cmap=cmap)
    ax.axhline(0, color='black', linewidth=1.5, linestyle='--')
    ax.axvline(0, color='black', linewidth=1.5, linestyle='--')
    ax.set_xlabel('Principal Component 1', fontsize=24)
    ax.set_ylabel('Principal Component 2', fontsize=24)
    ax.tick_params(axis='both', which='both', direction='in', labelsize=24,
                   top=True, bottom=True, left=True, right=True, length=8)
    ax.tick_params(axis='both', which='minor', direction='in', length=3, width=1.0)
    ax.minorticks_on()
    ax.xaxis.set_minor_locator(AutoMinorLocator(5))
    ax.yaxis.set_minor_locator(AutoMinorLocator(5))
    cbar = plt.colorbar(contour, ax=ax)
    cbar.set_label(label, fontsize=24)
    cbar.set_ticks(np.linspace(np.nanmin(zi), np.nanmax(zi), 6))
    cbar.ax.tick_params(labelsize=24)

def plot_contours(df: pd.DataFrame,
                  feature_cols: List[str],
                  save_base: str,
                  symbol_converter: SymbolConverter) -> None:
    available = [f for f in feature_cols if f in df.columns]
    if not available:
        Colors.warning("No feature columns for contour plots."); return

    x = df['PC1'].values; y = df['PC2'].values
    grids  = {feat: _build_contour_grid(x, y, df[feat].values) for feat in available}
    labels = {feat: symbol_converter.get_axis_label(feat)      for feat in available}

    n     = len(available)
    ncols = min(3, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 4.5 * nrows))
    axes_flat = np.array(axes).flatten()
    for idx, feat in enumerate(available):
        _draw_single_contour(axes_flat[idx], *grids[feat], labels[feat])
    for idx in range(n, len(axes_flat)):
        axes_flat[idx].set_visible(False)
    plt.suptitle("PCA Space — Property Contour Maps", fontsize=18, fontweight='bold', y=1.01)
    plt.tight_layout()
    out_png = f"{save_base}_Contours_Combined.png"
    out_pdf = f"{save_base}_Contours_Combined.pdf"
    plt.savefig(out_png, dpi=600, bbox_inches="tight")
    plt.savefig(out_pdf, dpi=600, bbox_inches="tight")
    plt.close()
    Colors.success(f"Combined contour panel saved: {out_png}  |  {out_pdf}")

# ============================================================================
# PLOT 8: Property Threshold Analysis  (BLOCK 4)
# ============================================================================
def plot_property_threshold_analysis(df: pd.DataFrame,
                                     save_base: str,
                                     symbol_converter: SymbolConverter,
                                     dt_col: str = 'DT') -> None:
    Colors.info("Generating property threshold analysis plot...")
    def gradient_fill(ax, x, y_min, y_max, cmap_name, vmin, vmax):
        if len(x) == 0:
            return
        norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
        cmap = cmap_name if callable(cmap_name) else plt.get_cmap(cmap_name)
        for j in range(len(x) - 1):
            x0, x1 = x[j], x[j+1]
            ylo = (y_min[j] + y_min[j+1]) / 2
            yhi = (y_max[j] + y_max[j+1]) / 2
            color = cmap(norm(x[j]))
            ax.fill_between([x0, x1], [ylo, ylo], [yhi, yhi], color=color, edgecolor='none')

    x_label = f"Debye Temperature Threshold ({COLUMN_UNITS.get(dt_col, 'K')})"
    
    #########################################################
    ## Update it as per the target property
    #########################################################
    dt_thresholds_low  = np.arange(-1000, 350 + 1, 2)
    dt_thresholds_high = np.arange(1000, df[dt_col].max() + 1, 2)

    input_columns = [c for c in DataLoader.FEATURE_COLUMNS
                 if c in df.columns and c != dt_col]

    if not input_columns:
        Colors.warning("No property columns for threshold analysis."); return

    y_labels = symbol_converter.labels_for_columns(input_columns)

    def collect_ranges(thresholds, mask_fn):
        out = {}
        for thr in thresholds:
            sub = df[mask_fn(thr)]
            if len(sub): out[thr] = sub[input_columns].describe().loc[['min', 'max']]
        return pd.concat(out, axis=0) if out else pd.DataFrame()

    df_low  = collect_ranges(dt_thresholds_low,  lambda t: df[dt_col] < t)
    df_high = collect_ranges(dt_thresholds_high, lambda t: df[dt_col] > t)
    if df_low.empty and df_high.empty:
        Colors.warning("No data for threshold analysis."); return

    df_low.index.names  = ['Threshold_K', 'Stat']
    df_high.index.names = ['Threshold_K', 'Stat']
    combined = pd.concat([df_low, df_high])

    fig, axes = plt.subplots(nrows=len(input_columns), ncols=1,
                              figsize=(16.5, 2.6 * len(input_columns)), sharex=True)
    if len(input_columns) == 1:
        axes = np.array([axes])

    for i, feature in enumerate(input_columns):
        ax = axes[i]
        if feature not in combined.columns: continue
        min_vals   = combined.xs('min', level='Stat')[feature]
        max_vals   = combined.xs('max', level='Stat')[feature]
        thresholds = min_vals.index
    
        #########################################################
        ## Update it as per the target property
        #########################################################        
        low_mask   = thresholds < 350
        mid_mask   = (thresholds >= 350) & (thresholds <= 1000)
        high_mask  = thresholds > 1000
        cmap_low  = LinearSegmentedColormap.from_list('#80acff',    ['#2276FE', '#92CFFF'])
        cmap_high = LinearSegmentedColormap.from_list('#fc7c82', ['#FF737F', '#FF2F3B'])
        ax.plot(thresholds[low_mask],  min_vals[low_mask],  color='blue',    linestyle='--', linewidth=2)
        ax.plot(thresholds[low_mask],  max_vals[low_mask],  color='darkblue', linestyle='-', linewidth=2)
        ax.fill_between(thresholds[mid_mask],  min_vals[mid_mask],  max_vals[mid_mask],  color='white',   edgecolor='none')
        ax.plot(thresholds[mid_mask],  min_vals[mid_mask],  color='black',   linestyle='--', linewidth=2)
        ax.plot(thresholds[mid_mask],  max_vals[mid_mask],  color='black',   linestyle='-',  linewidth=2)
        if np.any(low_mask):
                x_l = thresholds[low_mask].values
                gradient_fill(ax, x_l,
                              min_vals[low_mask].values,
                              max_vals[low_mask].values,
                              cmap_low,                        
                              vmin=x_l.min(), vmax=x_l.max())
        if np.any(high_mask):
                x_h = thresholds[high_mask].values
                gradient_fill(ax, x_h,
                              min_vals[high_mask].values,
                              max_vals[high_mask].values,
                              cmap_high,                      
                              vmin=x_h.min(), vmax=x_h.max())
        ax.plot(thresholds[high_mask], min_vals[high_mask], color='red',     linestyle='--', linewidth=2)
        ax.plot(thresholds[high_mask], max_vals[high_mask], color='darkred', linestyle='-',  linewidth=2)
        if np.any(low_mask) and np.any(mid_mask):
            le = thresholds[low_mask][-1]
            ax.vlines(x=le, ymin=min_vals[le], ymax=max_vals[le],
                      color='black', linestyle='-', linewidth=2)
        if np.any(high_mask) and np.any(mid_mask):
            hs = thresholds[high_mask][0]
            ax.vlines(x=hs, ymin=min_vals[hs], ymax=max_vals[hs],
                      color='black', linestyle='-', linewidth=2)
        ax.set_ylabel(y_labels[i], fontsize=25)
        ax.tick_params(labelsize=25, axis='both', which='both', direction='in',
                       length=10, top=True, bottom=True, left=True, right=True, width=1.8)
        ax.tick_params(axis='both', which='minor', direction='in', length=5, width=1.8,
                       top=True, bottom=True, left=True, right=True)
        ax.minorticks_on()
        ax.xaxis.set_minor_locator(AutoMinorLocator(15))
        ax.yaxis.set_minor_locator(AutoMinorLocator(5))
        ax.yaxis.set_major_locator(MaxNLocator(3))
        ax.xaxis.set_major_locator(MaxNLocator(8))
        ax.spines['top'].set_linewidth(2.0)
        ax.spines['bottom'].set_linewidth(2.0)
        ax.spines['left'].set_linewidth(2.0)
        ax.spines['right'].set_linewidth(2.0)
    max_handle = TwoColorLine('darkblue', 'darkred', linestyle='-',  linewidth=2.5)
    min_handle = TwoColorLine('blue',     'red',     linestyle='--', linewidth=2.5)
    fill_high  = Patch(color='#fc7c82')
    fill_low   = Patch(color='#80acff')
    dt_sym     = symbol_converter.get(dt_col)
    axes[0].legend(
        handles=[max_handle, min_handle, fill_high, fill_low],
        labels=['Max Value Line', 'Min Value Line',
                f'Higher {dt_sym} Range', f'Lower {dt_sym} Range'],
        handler_map={TwoColorLine: HandlerTwoColorLine(gap_ratio=0.08)},
        loc='upper left', fontsize=15, frameon=True, edgecolor='black',
    )
    axes[-1].set_xlabel(x_label, fontsize=25, labelpad=12)
    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    out_png = f"{save_base}_Property_Threshold_Analysis.png"
    out_pdf = f"{save_base}_Property_Threshold_Analysis.pdf"
    plt.savefig(out_png, dpi=600, bbox_inches="tight")
    plt.savefig(out_pdf, dpi=600, bbox_inches="tight")
    plt.close()
    Colors.success(f"Property threshold analysis saved: {out_png}  |  {out_pdf}")

    #########################################################
    ## Update it as per the target property
    #########################################################
    low_mat  = df[df[dt_col] < 350]
    high_mat = df[df[dt_col] >= 1000]
    print(f"\n{Colors.CYAN}Threshold Statistics:{Colors.ENDC}")
    print(f"  Materials in Low  {dt_col} regime (<350 K):  {len(low_mat):,}")
    print(f"  Materials in High {dt_col} regime (≥1000 K): {len(high_mat):,}\n")

def export_to_excel(df_pca, loadings, runner, feature_cols, symbol_converter, save_base, dt_col='DT', timing_log=None, n_cpus=1, backend_label='N/A'):

    Colors.info("Preparing Excel export...")

    wb = openpyxl.Workbook()
    wb.remove(wb.active) 

    # ── colour palette for sheet tabs ──────────────────────────────────────
    TAB_COLORS = [
        "1F4E79", "2E75B6", "2F5496", "375623", "548235",
        "70AD47", "833C00", "C55A11", "7B7B7B", "595959",
        "203864", "BF9000",
    ]

    # ── helper: create sheet ───────────────────────────────────────────────
    def make_ws(title: str, tab_idx: int) -> openpyxl.worksheet.worksheet.Worksheet:
        ws = wb.create_sheet(title=title)
        ws.sheet_properties.tabColor = TAB_COLORS[tab_idx % len(TAB_COLORS)]
        # Title row above header
        ws.row_dimensions[1].height = 18
        return ws

    # ── header fill colours per block ──────────────────────────────────────
    FILLS = {
        'raw':    ("1F4E79", "FFFFFF"),   # dark navy / white
        'pca':    ("2E75B6", "FFFFFF"),   # medium blue / white
        'stats':  ("375623", "FFFFFF"),   # dark green / white
        'dist':   ("833C00", "FFFFFF"),   # dark orange / white
        'thresh': ("7B7B7B", "FFFFFF"),   # grey / white
        'corr':   ("2F5496", "FFFFFF"),   # indigo / white
        'contrib':("203864", "FFFFFF"),   # midnight blue / white
    }

    pc_cols_present = [c for c in df_pca.columns if c.startswith('PC')]
    raw_cols        = [c for c in df_pca.columns if not c.startswith('PC')]

    # ══════════════════════════════════════════════════════════════════════
    # SHEET 01 – Raw Data
    # ══════════════════════════════════════════════════════════════════════
    ws1 = make_ws("01 Raw Data", 0)
    df_raw = df_pca[raw_cols].copy()
    _write_df_to_sheet(ws1, df_raw, header_fill=FILLS['raw'][0],
                       header_font=FILLS['raw'][1], num_fmt='0.0000')
    Colors.success("Sheet 01 – Raw Data")

    # ══════════════════════════════════════════════════════════════════════
    # SHEET 02 – PCA Scores
    # ══════════════════════════════════════════════════════════════════════
    ws2 = make_ws("02 PCA Scores", 1)
    id_cols = [c for c in ['material_id', 'formula', 'Category'] if c in df_pca.columns]
    df_scores = df_pca[id_cols + pc_cols_present].copy()
    _write_df_to_sheet(ws2, df_scores, header_fill=FILLS['pca'][0],
                       header_font=FILLS['pca'][1], num_fmt='0.00000')
    Colors.success("Sheet 02 – PCA Scores")

    # ══════════════════════════════════════════════════════════════════════
    # SHEET 03 – Explained Variance
    # ══════════════════════════════════════════════════════════════════════
    ws3 = make_ws("03 Explained Variance", 2)
    evr = runner.explained_variance
    cum = np.cumsum(evr)
    df_ev = pd.DataFrame({
        'Principal Component': [f'PC{i+1}' for i in range(len(evr))],
        'Explained Variance':  evr,
        'Cumulative Variance': cum,
        'Explained Variance (%)': evr * 100,
        'Cumulative Variance (%)': cum * 100,
    })
    _write_df_to_sheet(ws3, df_ev, header_fill=FILLS['pca'][0],
                       header_font=FILLS['pca'][1], num_fmt='0.0000')
    Colors.success("Sheet 03 – Explained Variance")

    # ══════════════════════════════════════════════════════════════════════
    # SHEET 04 – PC Loadings
    # ══════════════════════════════════════════════════════════════════════
    ws4 = make_ws("04 PC Loadings", 3)
    n_pcs = loadings.shape[1]
    df_load = pd.DataFrame(loadings,
                            index=feature_cols,
                            columns=[f'PC{i+1}' for i in range(n_pcs)]).reset_index()
    df_load.rename(columns={'index': 'Feature'}, inplace=True)
    df_load.insert(1, 'Label',
                   [symbol_converter.get(c) for c in feature_cols])
    _write_df_to_sheet(ws4, df_load, header_fill=FILLS['pca'][0],
                       header_font=FILLS['pca'][1], num_fmt='0.00000')
    Colors.success("Sheet 04 – PC Loadings")

    # ══════════════════════════════════════════════════════════════════════
    # SHEET 05 – Category Summary
    # ══════════════════════════════════════════════════════════════════════
    ws5 = make_ws("05 Category Summary", 4)
    total = len(df_pca)
    rows_cat = []
    dt_avail = dt_col in df_pca.columns
    for cat in CategoryConfig.DISPLAY_ORDER:
        sub = df_pca[df_pca['Category'] == cat]
        if sub.empty:
            continue
        row = {
            'Category': cat,
            'Count': len(sub),
            'Share (%)': round(len(sub) / total * 100, 2),
        }
        if dt_avail:
            row[f'{dt_col} Min (K)']  = sub[dt_col].min()
            row[f'{dt_col} Max (K)']  = sub[dt_col].max()
            row[f'{dt_col} Mean (K)'] = sub[dt_col].mean()
            row[f'{dt_col} Std (K)']  = sub[dt_col].std()
        rows_cat.append(row)
    # totals row
    rows_cat.append({
        'Category': 'TOTAL', 'Count': total, 'Share (%)': 100.0,
    })
    df_cat = pd.DataFrame(rows_cat)
    _write_df_to_sheet(ws5, df_cat, header_fill=FILLS['stats'][0],
                       header_font=FILLS['stats'][1], num_fmt='0.00')
    # colour-code category names
    cat_col_idx = 1
    for row_idx in range(2, ws5.max_row + 1):
        cat_name = ws5.cell(row=row_idx, column=cat_col_idx).value
        hex_fill = CategoryConfig.XL_HEADER_FILLS.get(cat_name)
        if hex_fill:
            ws5.cell(row=row_idx, column=cat_col_idx).fill = \
                PatternFill("solid", fgColor=hex_fill)
            # Ensure dark text for readability
            ws5.cell(row=row_idx, column=cat_col_idx).font = \
                Font(name='Arial', size=10, bold=True, color="000000")
    Colors.success("Sheet 05 – Category Summary")

    # ══════════════════════════════════════════════════════════════════════
    # SHEET 06 – Feature Statistics
    # ══════════════════════════════════════════════════════════════════════
    ws6 = make_ws("06 Feature Statistics", 5)
    stat_rows = []
    for cat in CategoryConfig.DISPLAY_ORDER:
        sub = df_pca[df_pca['Category'] == cat]
        if sub.empty:
            continue
        for feat in feature_cols:
            if feat not in sub.columns:
                continue
            unit = COLUMN_UNITS.get(feat, '')
            stat_rows.append({
                'Category':    cat,
                'Feature':     feat,
                'Label':       symbol_converter.get(feat),
                'Unit':        unit,
                'Min':         sub[feat].min(),
                'Max':         sub[feat].max(),
                'Mean':        sub[feat].mean(),
                'Median':      sub[feat].median(),
                'Std':         sub[feat].std(),
                'Count':       sub[feat].count(),
            })
    df_fstats = pd.DataFrame(stat_rows)
    _write_df_to_sheet(ws6, df_fstats, header_fill=FILLS['stats'][0],
                       header_font=FILLS['stats'][1], num_fmt='0.0000')
    Colors.success("Sheet 06 – Feature Statistics")

    # ══════════════════════════════════════════════════════════════════════
    # SHEET 07 – PC1 Bin Stats
    # ══════════════════════════════════════════════════════════════════════
    ws7 = make_ws("07 PC1 Bin Stats", 6)
    df_pc1_bins = _compute_pc_bin_stats(df_pca, 'PC1', dt_col, bin_size=0.950)
    df_pc1_bins.rename(columns={
        'count': 'N_materials',
        'DT_mean': f'{dt_col}_mean (K)',
        'DT_std':  f'{dt_col}_std (K)',
        'DT_min':  f'{dt_col}_min (K)',
        'DT_max':  f'{dt_col}_max (K)',
        'DT_p10':  f'{dt_col}_p10 (K)',
        'DT_p90':  f'{dt_col}_p90 (K)',
    }, inplace=True)
    _write_df_to_sheet(ws7, df_pc1_bins, header_fill=FILLS['dist'][0],
                       header_font=FILLS['dist'][1], num_fmt='0.0000')
    Colors.success("Sheet 07 – PC1 Bin Stats")

    # ══════════════════════════════════════════════════════════════════════
    # SHEET 08 – PC2 Bin Stats
    # ══════════════════════════════════════════════════════════════════════
    ws8 = make_ws("08 PC2 Bin Stats", 7)
    df_pc2_bins = _compute_pc_bin_stats(df_pca, 'PC2', dt_col, bin_size=0.475)
    df_pc2_bins.rename(columns={
        'count': 'N_materials',
        'DT_mean': f'{dt_col}_mean (K)',
        'DT_std':  f'{dt_col}_std (K)',
        'DT_min':  f'{dt_col}_min (K)',
        'DT_max':  f'{dt_col}_max (K)',
        'DT_p10':  f'{dt_col}_p10 (K)',
        'DT_p90':  f'{dt_col}_p90 (K)',
    }, inplace=True)
    _write_df_to_sheet(ws8, df_pc2_bins, header_fill=FILLS['dist'][0],
                       header_font=FILLS['dist'][1], num_fmt='0.0000')
    Colors.success("Sheet 08 – PC2 Bin Stats")

    # ══════════════════════════════════════════════════════════════════════
    # SHEET 09 – DT Distribution
    # ══════════════════════════════════════════════════════════════════════
    ws9 = make_ws("09 DT Distribution", 8)
    df_dt_dist = _compute_dt_distribution(df_pca, dt_col, bin_size=75)
    _write_df_to_sheet(ws9, df_dt_dist, header_fill=FILLS['dist'][0],
                       header_font=FILLS['dist'][1], num_fmt='0.00')
    Colors.success("Sheet 09 – DT Distribution")

    # ══════════════════════════════════════════════════════════════════════
    # SHEET 10 – Threshold Low  (DT < 350 K)
    # ══════════════════════════════════════════════════════════════════════
    ws10 = make_ws("10 Threshold Low (<350K)", 9)
    df_thresh_low, df_thresh_high = _compute_threshold_ranges(
        df_pca, feature_cols, dt_col, low_cut=350, high_cut=1000)
    _write_df_to_sheet(ws10, df_thresh_low, header_fill=FILLS['thresh'][0],
                       header_font=FILLS['thresh'][1], num_fmt='0.0000')
    Colors.success("Sheet 10 – Threshold Low")

    # ══════════════════════════════════════════════════════════════════════
    # SHEET 11 – Threshold High  (DT > 1000 K)
    # ══════════════════════════════════════════════════════════════════════
    ws11 = make_ws("11 Threshold High (>1000K)", 10)
    _write_df_to_sheet(ws11, df_thresh_high, header_fill=FILLS['thresh'][0],
                       header_font=FILLS['thresh'][1], num_fmt='0.0000')
    Colors.success("Sheet 11 – Threshold High")

    # ══════════════════════════════════════════════════════════════════════
    # SHEET 12 – Correlation Matrix
    # ══════════════════════════════════════════════════════════════════════
    ws12 = make_ws("12 Correlation Matrix", 11)
    corr_cols = [c for c in feature_cols + pc_cols_present if c in df_pca.columns]
    df_corr   = df_pca[corr_cols].corr()
    df_corr_out = df_corr.reset_index().rename(columns={'index': 'Feature'})
    _write_df_to_sheet(ws12, df_corr_out, header_fill=FILLS['corr'][0],
                       header_font=FILLS['corr'][1], num_fmt='0.0000')

    for row_idx in range(2, ws12.max_row + 1):
        for col_idx in range(2, ws12.max_column + 1):
            cell = ws12.cell(row=row_idx, column=col_idx)
            val  = cell.value
            if not isinstance(val, (float, int, np.floating)):
                continue
            # Map [-1, 1] → colour
            t = (float(val) + 1) / 2          # 0 = neg, 0.5 = neutral, 1 = pos
            r = int(255 * (1 - t))
            g = int(255 * t)
            b = 120
            hex_col = f"{r:02X}{g:02X}{b:02X}"
            cell.fill = PatternFill("solid", fgColor=hex_col)
            cell.font = Font(name='Arial', size=9,
                             color="000000" if 0.25 < t < 0.75 else "FFFFFF")
    Colors.success("Sheet 12 – Correlation Matrix (heatmap)")

    # ══════════════════════════════════════════════════════════════════════
    # SHEET 13 – PCA Contributions  (% variance per feature per PC)
    # ══════════════════════════════════════════════════════════════════════
    ws13 = make_ws("13 PCA Contributions", 0)
    # Contribution = squared loading × explained variance of PC (× 100 for %)
    contrib = (loadings ** 2) * runner.explained_variance[np.newaxis, :] * 100
    df_contrib = pd.DataFrame(contrib,
                               index=feature_cols,
                               columns=[f'PC{i+1} contrib (%)' for i in range(n_pcs)])
    df_contrib['Total contrib (%)'] = df_contrib.sum(axis=1)
    df_contrib.insert(0, 'Label',
                      [symbol_converter.get(c) for c in feature_cols])
    df_contrib = df_contrib.reset_index().rename(columns={'index': 'Feature'})
    _write_df_to_sheet(ws13, df_contrib, header_fill=FILLS['contrib'][0],
                       header_font=FILLS['contrib'][1], num_fmt='0.000')
    max_contrib = float(df_contrib[[c for c in df_contrib.columns
                                     if 'contrib' in c]].max().max())
    for row_idx in range(2, ws13.max_row + 1):
        for col_idx in range(3, ws13.max_column + 1):
            cell = ws13.cell(row=row_idx, column=col_idx)
            val  = cell.value
            if not isinstance(val, (float, int, np.floating)) or max_contrib == 0:
                continue
            intensity = min(1.0, float(val) / max_contrib)
            b_val = int(220 - intensity * 130)
            g_val = int(235 - intensity * 50)
            hex_col = f"00{g_val:02X}{b_val:02X}"
            cell.fill = PatternFill("solid", fgColor=hex_col)
    Colors.success("Sheet 13 – PCA Contributions")

    # ══════════════════════════════════════════════════════════════════════
    # SHEET 14 – Timing Report
    # ══════════════════════════════════════════════════════════════════════
    if timing_log:
        ws14 = make_ws("14 Timing Report", 11)
    
        total = sum(v for k, v in timing_log.items() if k != 'TOTAL wall-clock')
        df_timing = pd.DataFrame([
            {
                'Phase': k,
                'Time (s)': round(v, 4),
                'Time (mm:ss)': f"{int(v//60):02d}:{v%60:05.2f}",
                'Share of total (%)': round(v / timing_log.get('TOTAL wall-clock', v) * 100, 2),
            }
            for k, v in timing_log.items()
        ])
        _write_df_to_sheet(ws14, df_timing, header_fill="2F4F4F",
                       header_font="FFFFFF", num_fmt='0.0000')

        try:
            cpu_count_physical = psutil.cpu_count(logical=False) if psutil else 'N/A'
            cpu_count_logical  = psutil.cpu_count(logical=True)  if psutil else 'N/A'
            ram_gb             = round(psutil.virtual_memory().total / 1e9, 2) if psutil else 'N/A'
        except Exception:
            cpu_count_physical = cpu_count_logical = ram_gb = 'N/A'

        try:
            import sklearn, scipy, openpyxl as oxl, seaborn as sns_mod
            sklearn_ver  = sklearn.__version__
            scipy_ver    = scipy.__version__
            openpyxl_ver = oxl.__version__
            seaborn_ver  = sns_mod.__version__
        except Exception:
            sklearn_ver = scipy_ver = openpyxl_ver = seaborn_ver = 'N/A'

        try:
            import plotly
            plotly_ver = plotly.__version__
        except ImportError:
            plotly_ver = 'not installed'

        try:
            slurm_job    = os.environ.get('SLURM_JOB_ID',       'N/A')
            slurm_node   = os.environ.get('SLURM_NODELIST',     'N/A')
            slurm_cpus   = os.environ.get('SLURM_CPUS_PER_TASK','N/A')
            slurm_mem    = os.environ.get('SLURM_MEM_PER_NODE', 'N/A')
            slurm_part   = os.environ.get('SLURM_JOB_PARTITION','N/A')
        except Exception:
            slurm_job = slurm_node = slurm_cpus = slurm_mem = slurm_part = 'N/A'

        sys_info = [
            ('── Run Info ──',               ''),
            ('Run timestamp',                datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')),
            ('Total materials',              len(df_pca)),
            ('Features used',                ', '.join(feature_cols)),
            ('CPUs requested (this run)',    n_cpus),
            ('Backend used',                 backend_label),
            ('',                             ''),
            ('── System Info ──',            ''),
            ('OS',                           platform.system() + ' ' + platform.release()),
            ('Hostname',                     platform.node()),
            ('CPU model',                    platform.processor() or 'N/A'),
            ('Physical CPU cores',           cpu_count_physical),
            ('Logical CPU cores',            cpu_count_logical),
            ('Total RAM (GB)',               ram_gb),
            ('',                             ''),
            ('── SLURM Info ──',             ''),
            ('SLURM Job ID',                 slurm_job),
            ('SLURM Node',                   slurm_node),
            ('SLURM CPUs per task',          slurm_cpus),
            ('SLURM Memory per node (MB)',   slurm_mem),
            ('SLURM Partition',              slurm_part),
            ('',                             ''),
            ('── Python & Library Versions ──', ''),
            ('Python',          platform.python_version()),
            ('NumPy',           np.__version__),
            ('Pandas',          pd.__version__),
            ('Matplotlib',      matplotlib.__version__),
            ('Scikit-learn',    sklearn_ver),
            ('SciPy',           scipy_ver),
            ('Seaborn',         seaborn_ver),
            ('Openpyxl',        openpyxl_ver),
            ('Plotly',          plotly_ver),
            ('Psutil',          psutil.__version__ if psutil else 'not installed'),
        ]

        info_row = len(df_timing) + 3
        for offset, (label, value) in enumerate(sys_info):
            cell_label = ws14.cell(row=info_row + offset, column=1, value=str(label))
            cell_value = ws14.cell(row=info_row + offset, column=2, value=str(value))
            if label.startswith('──'):
                cell_label.font = Font(bold=True, name='Arial', size=10, color='FFFFFF')
                cell_label.fill = PatternFill("solid", fgColor="2F4F4F")
                cell_value.fill = PatternFill("solid", fgColor="2F4F4F")
            else:
                cell_label.font = Font(bold=True, name='Arial', size=10)
                cell_value.font = Font(name='Arial', size=10)

        # Heat-map the Share column (col 4): deeper green = longer
        max_share = df_timing['Share of total (%)'].max()
        for row_idx in range(2, len(df_timing) + 2):
            cell = ws14.cell(row=row_idx, column=4)
            val  = cell.value
            if isinstance(val, (float, int)) and max_share > 0:
                intensity = min(1.0, float(val) / max_share)
                g_val = int(200 - intensity * 100)
                cell.fill = PatternFill("solid", fgColor=f"00{g_val:02X}00")
                cell.font = Font(name='Arial', size=10,
                                 color="FFFFFF" if intensity > 0.5 else "000000")
        Colors.success("Sheet 14 – Timing Report")

    # ── Save ──────────────────────────────────────────────────────────────
    xl_path = f"{save_base}_PCA_Export.xlsx"
    wb.save(xl_path)
    Colors.success(f"Excel workbook saved → {xl_path}")
    print(f"\n  {Colors.BOLD}Sheets exported:{Colors.ENDC}")
    for ws in wb.worksheets:
        print(f"    {Colors.GREEN}•{Colors.ENDC}  {ws.title}")
    print()
    return xl_path

def export_figure_source_data(df_pca: pd.DataFrame,
                               loadings: np.ndarray,
                               runner: 'PCARunner',
                               feature_cols: List[str],
                               symbol_converter: 'SymbolConverter',
                               save_base: str,
                               dt_col: str = 'DT') -> str:

    Colors.info("Preparing Figure Source Data Excel export...")
    wb = openpyxl.Workbook()
    wb.remove(wb.active)

    FILL = ("2F4F4F", "FFFFFF")  

    def make_ws(title):
        ws = wb.create_sheet(title=title)
        return ws

    pc_cols_present = [c for c in df_pca.columns if c.startswith('PC')]

    # ── Fig S1: Explained Variance ────────────────────────────────────────
    ws = make_ws("Fig_ExplainedVariance")
    evr = runner.explained_variance
    df_ev = pd.DataFrame({
        'Principal_Component': [f'PC{i+1}' for i in range(len(evr))],
        'Explained_Variance':  evr,
        'Cumulative_Variance': np.cumsum(evr),
    })
    _write_df_to_sheet(ws, df_ev, header_fill=FILL[0], header_font=FILL[1])

    # ── Fig S2: PC Loadings biplot ────────────────────────────────────────
    ws = make_ws("Fig_PC_Loadings")
    n_pcs = loadings.shape[1]
    df_load = pd.DataFrame(loadings, index=feature_cols,
                            columns=[f'PC{i+1}_loading' for i in range(n_pcs)])
    df_load.insert(0, 'Feature', feature_cols)
    df_load.insert(1, 'Label', [symbol_converter.get(c) for c in feature_cols])
    _write_df_to_sheet(ws, df_load.reset_index(drop=True),
                       header_fill=FILL[0], header_font=FILL[1])

    # ── Fig S3: PCA Scatter (PC1 vs PC2 raw scores) ───────────────────────
    ws = make_ws("Fig_PCA_Scatter")
    id_cols = [c for c in ['material_id', 'formula', 'Category'] if c in df_pca.columns]
    df_scatter = df_pca[id_cols + pc_cols_present + feature_cols].copy()
    _write_df_to_sheet(ws, df_scatter, header_fill=FILL[0], header_font=FILL[1])

    # ── Fig S4: Material Count vs PC1/PC2 ────────────────────────────────
    ws = make_ws("Fig_Count_vs_PC1_PC2")
    sorted_data = df_pca.sort_values("PC1").reset_index(drop=True)
    bin_size_PC1, bin_size_PC2 = 0.950, 0.475
    bins_PC1 = np.arange(sorted_data["PC1"].min(),
                          sorted_data["PC1"].max() + bin_size_PC1, bin_size_PC1)
    bins_PC2 = np.arange(sorted_data["PC2"].min(),
                          sorted_data["PC2"].max() + bin_size_PC2, bin_size_PC2)

    def _bin_count_src(data, pc_col, bins, dt_col):
        grp = (data.groupby(pd.cut(data[pc_col], bins=bins))
               .agg(mean_pc=(pc_col, 'mean'), count=(dt_col, 'size'))
               .reset_index(drop=True))
        grp.rename(columns={'mean_pc': pc_col, 'count': 'N_materials'}, inplace=True)
        return grp

    df_cnt_pc1 = _bin_count_src(sorted_data, 'PC1', bins_PC1, dt_col)
    df_cnt_pc2 = _bin_count_src(sorted_data, 'PC2', bins_PC2, dt_col)
    max_len = max(len(df_cnt_pc1), len(df_cnt_pc2))
    df_cnt_pc1 = df_cnt_pc1.reindex(range(max_len))
    df_cnt_pc2 = df_cnt_pc2.reindex(range(max_len))
    df_cnt_combined = pd.concat(
        [df_cnt_pc1.add_suffix('_PC1bin'), df_cnt_pc2.add_suffix('_PC2bin')], axis=1)
    _write_df_to_sheet(ws, df_cnt_combined, header_fill=FILL[0], header_font=FILL[1])

    # ── Fig S5: Theta_D Stats vs PC1/PC2 ─────────────────────────────────
    ws = make_ws("Fig_ThetaD_Stats_vs_PC")

    def _agg_stats_src(data, pc_col, bins, dt_col):
        grp = data.groupby(pd.cut(data[pc_col], bins=bins)).agg(
            mean_pc    = (pc_col, 'mean'),
            mean_DT    = (dt_col, 'mean'),
            std_DT     = (dt_col, 'std'),
            q10_DT     = (dt_col, lambda x: np.percentile(x, 10) if len(x) else np.nan),
            q90_DT     = (dt_col, lambda x: np.percentile(x, 90) if len(x) else np.nan),
            count      = (dt_col, 'size'),
        ).reset_index(drop=True)
        grp.rename(columns={'mean_pc': pc_col}, inplace=True)
        return grp

    df_s_pc1 = _agg_stats_src(sorted_data, 'PC1', bins_PC1, dt_col)
    df_s_pc2 = _agg_stats_src(sorted_data, 'PC2', bins_PC2, dt_col)
    max_len = max(len(df_s_pc1), len(df_s_pc2))
    df_s_pc1 = df_s_pc1.reindex(range(max_len))
    df_s_pc2 = df_s_pc2.reindex(range(max_len))
    df_stats_combined = pd.concat(
        [df_s_pc1.add_suffix('_PC1'), df_s_pc2.add_suffix('_PC2')], axis=1)
    _write_df_to_sheet(ws, df_stats_combined, header_fill=FILL[0], header_font=FILL[1])

    # ── Fig S6: Theta_D Distribution ──────────────────────────────────────
    ws = make_ws("Fig_ThetaD_Distribution")
    df_dt_dist = _compute_dt_distribution(df_pca, dt_col, bin_size=75)
    _write_df_to_sheet(ws, df_dt_dist, header_fill=FILL[0], header_font=FILL[1])

    # ── Fig S7: Contour source (raw PC scores + property values) ─────────
    ws_raw = make_ws("Fig_Contours_RawScores")
    id_cols_contour = [c for c in ['material_id', 'formula', 'Category'] if c in df_pca.columns]
    contour_cols    = id_cols_contour + ['PC1', 'PC2'] + [f for f in feature_cols if f in df_pca.columns]
    _write_df_to_sheet(ws_raw, df_pca[contour_cols].copy(),
                       header_fill=FILL[0], header_font=FILL[1])
    # ── Fig S7b: Full interpolation grids (one sheet per feature) ────────
    x = df_pca['PC1'].values
    y = df_pca['PC2'].values
    resolution = 150
    xi_1d = np.linspace(x.min(), x.max(), resolution)
    yi_1d = np.linspace(y.min(), y.max(), resolution)
    xi_grid, yi_grid = np.meshgrid(xi_1d, yi_1d)

    for feat in [f for f in feature_cols if f in df_pca.columns]:
        ws_interp = make_ws(f"Fig_Contour_{feat}_Grid")
        z = df_pca[feat].values
        zi = griddata((x, y), z, (xi_grid, yi_grid), method='linear')
        zi_clipped = np.clip(zi, np.nanmin(z), np.nanmax(z))
        df_grid = pd.DataFrame(
                        zi_clipped,
                        index=[float(v) for v in np.round(yi_1d, 2)],
                        columns=[float(v) for v in np.round(xi_1d, 2)])
        df_grid_out = df_grid.reset_index()
        df_grid_out.rename(columns={'index': 'PC2_PC1'}, inplace=True)
        _write_df_to_sheet(ws_interp, df_grid_out,
                           header_fill=FILL[0], header_font=FILL[1],
                           num_fmt='0.000000')
        Colors.success(f"  Contour grid saved: {feat} ({resolution}×{resolution})")
    # ── Fig S8: Property Threshold Analysis ───────────────────────────────
    ws_low  = make_ws("Fig_Threshold_Low_Source")
    ws_high = make_ws("Fig_Threshold_High_Source")
    input_columns = [c for c in DataLoader.FEATURE_COLUMNS
                     if c in df_pca.columns and c != dt_col]
    df_thresh_low, df_thresh_high = _compute_threshold_ranges(
        df_pca, input_columns, dt_col, low_cut=350, high_cut=1000)
    _write_df_to_sheet(ws_low,  df_thresh_low,  header_fill=FILL[0], header_font=FILL[1])
    _write_df_to_sheet(ws_high, df_thresh_high, header_fill=FILL[0], header_font=FILL[1])

    # ── Save ──────────────────────────────────────────────────────────────
    src_path = f"{save_base}_Figure_Source_Data.xlsx"
    wb.save(src_path)
    Colors.success(f"Figure source data saved → {src_path}")
    print(f"\n  {Colors.BOLD}Figure source sheets:{Colors.ENDC}")
    for ws in wb.worksheets:
        print(f"    {Colors.GREEN}•{Colors.ENDC}  {ws.title}")
    print()
    return src_path

# ============================================================================
# SUMMARY PRINTER
# ============================================================================
def print_category_summary(df: pd.DataFrame) -> None:
    Colors.header("CATEGORY SUMMARY")
    counts = df["Category"].value_counts()
    total  = len(df)
    col_w  = max(len(c) for c in counts.index) + 2

    print(f"  {Colors.BOLD}{'Category':<{col_w}}  {'Count':>8}  {'Share':>7}   Distribution{Colors.ENDC}")
    print(f"  {'-'*col_w}  {'-'*8}  {'-'*7}   {'-'*40}")

    cat_colors = {
        "Single Element":              Colors.CYAN,
        "Metal & Metalloid Compounds": Colors.YELLOW,
        "Oxides":                      Colors.YELLOW,
        "Nitrides":                    Colors.HEADER,
        "Halides":                     Colors.GREEN,
        "Borides":                     Colors.CYAN,
        "Carbides":                    Colors.RED,
        "Other Compounds":             Colors.BLUE,
    }
    try:
        '█'.encode(sys.stdout.encoding or 'utf-8')
        BAR = '█'
    except (UnicodeEncodeError, LookupError):
        BAR = '#'

    for cat in CategoryConfig.DISPLAY_ORDER:
        if cat not in counts: continue
        count = counts[cat]
        pct   = count / total * 100
        bar   = BAR * max(1, int(pct / 2.5))
        col   = cat_colors.get(cat, Colors.CYAN)
        print(f"  {col}{cat:<{col_w}}{Colors.ENDC}"
              f"  {count:>8,}  {pct:>6.1f}%   {col}{bar}{Colors.ENDC}")

    print(f"\n  {'-'*70}")
    print(f"  {Colors.BOLD}Total: {total:,} materials{Colors.ENDC}\n")
# ============================================================================
# MAIN
# ============================================================================
def main():
    _timing_log = {}
    _t_pipeline = time.perf_counter()
    Colors.header("PCA ANALYSIS OF MATERIALS DATASET")

    # ── Detect environment first ───────────────────────────────────────────
    env = _detect_environment()
    pd.set_option('mode.copy_on_write', True)

    # ── CPU selection ──────────────────────────────────────────────────────
    print(f"\n{Colors.CYAN}  vCPUs available: {env['machine_cpus']}"
          f"{'  |  SLURM allocation: ' + str(env['slurm_cpus']) if env['in_slurm'] else ''}")
    print(f"\n  How many vCPU (virtual cores) to use? "
          f"(max: {env['machine_cpus']})\n"
          f"  Press [Enter] to use 1 (safe default), or enter a number: "
          f"{Colors.ENDC}", end='')

    _cpu_input = input().strip()
    if _cpu_input.isdigit() and 1 <= int(_cpu_input) <= env['machine_cpus']:
        n_cpus = int(_cpu_input)
    elif _cpu_input == '':
        n_cpus = 1
    else:
        Colors.warning("Invalid input — defaulting to 1 vCPU.")
        n_cpus = 1

    # ── Select backend automatically based on environment ─────────────────
    backend_cfg = _select_backend(n_cpus, env)
    Colors.success(f"Using {n_cpus} vCPU(s)  |  Backend: {backend_cfg['label']}")

    # ── STEP 1: Select data folder ─────────────────────────────────────────
    data_folder = FileManager.select_data_folder()

    # ── STEP 2a: Preview files and let user pick a subset ──────────────────
    Colors.step(2, "SELECT FILES & FEATURES")
    xlsx_files = sorted([f for f in os.listdir(data_folder) if f.endswith(".xlsx")])
    if not xlsx_files:
        Colors.error(f"No .xlsx files found in: {data_folder}"); sys.exit(1)

    print(f"\n{Colors.BOLD}  Files found:{Colors.ENDC}")
    for i, f in enumerate(xlsx_files, 1):
        print(f"    {Colors.GREEN}[{i}]{Colors.ENDC}  {f}")

    _sel = input(
        f"\n{Colors.CYAN}  Load ALL files? Press [Enter], or enter serial numbers/filenames "
        f"(comma-separated, e.g. '1,3' or 'oxide.xlsx, nitride.xlsx'): {Colors.ENDC}"
    ).strip()

    if _sel:
        _tokens = [t.strip() for t in _sel.split(',') if t.strip()]
        _selected = []
        for tok in _tokens:
            if tok.isdigit():
                idx = int(tok) - 1
                if 0 <= idx < len(xlsx_files):
                    _selected.append(xlsx_files[idx])
                else:
                    Colors.warning(f"Serial number {tok} out of range — skipped.")
            else:
                if tok in xlsx_files:
                    _selected.append(tok)
                else:
                    Colors.warning(f"File '{tok}' not found — skipped.")
        _file_subset = _selected if _selected else None
    else:
        _file_subset = None

    # ── STEP 2b: Preview feature columns and let user pick a subset ────────
    _peek_file = os.path.join(data_folder, xlsx_files[0])
    try:
        _peek_df = pd.read_excel(_peek_file, nrows=0)
        _available_feats = [c for c in DataLoader.FEATURE_COLUMNS if c in _peek_df.columns]
    except Exception:
        _available_feats = DataLoader.FEATURE_COLUMNS

    print(f"\n{Colors.BOLD}  Available feature columns:{Colors.ENDC}")
    for _i, _c in enumerate(_available_feats, 1):
        print(f"    {Colors.GREEN}[{_i}]{Colors.ENDC}  {_c}  ({COLUMN_UNITS.get(_c, 'no unit')})")

    _feat_sel = input(
        f"\n{Colors.CYAN}  Use ALL features? Press [Enter], or enter serial numbers/column names "
        f"(comma-separated, e.g. '1,3,5' or 'SM,YM,DT'): {Colors.ENDC}"
    ).strip()

    if _feat_sel:
        _tokens = [t.strip() for t in _feat_sel.split(',') if t.strip()]
        _chosen = []
        for tok in _tokens:
            if tok.isdigit():
                idx = int(tok) - 1
                if 0 <= idx < len(_available_feats):
                    _chosen.append(_available_feats[idx])
                else:
                    Colors.warning(f"Serial number {tok} out of range — skipped.")
            else:
                if tok in _available_feats:
                    _chosen.append(tok)
                else:
                    Colors.warning(f"Column '{tok}' not found — skipped.")
        feature_cols = _chosen if _chosen else _available_feats
    else:
        feature_cols = _available_feats

    # ── STEP 3: Output folder ──────────────────────────────────────────────
    save_base = FileManager.select_output_location()

    # ── STEP 4: Symbol conversion file ────────────────────────────────────
    symbol_file = FileManager.select_symbol_file()
    sym         = SymbolConverter(symbol_file)

    # ── STEP 5: Interactive HTML scatter ─────────────────────────────────
    _use_interactive = False
    _inject_path     = None
    _sym_html        = None

    if PLOTLY_AVAILABLE:
        Colors.step(5, "INTERACTIVE SCATTER PLOT")
        _ans_interactive = input(
            f"\n{Colors.CYAN}  Generate interactive HTML scatter plot? (y/n) [Default: n]: {Colors.ENDC}"
        ).strip().lower()

        if _ans_interactive in ('y', 'yes'):

            # ── Inject file ───────────────────────────────────────────────
            while True:
                _inject_raw  = input(
                    f"{Colors.CYAN}  Enter full path or folder containing "
                    f"pca_scatter_inject.html: {Colors.ENDC}"
                ).strip().strip('"\'')
                _inject_base = os.path.abspath(os.path.expanduser(_inject_raw))

                if os.path.isfile(_inject_base):
                    _inject_path = _inject_base
                elif os.path.isdir(_inject_base):
                    _candidate = os.path.join(_inject_base, 'pca_scatter_inject.html')
                    if os.path.isfile(_candidate):
                        _inject_path = _candidate
                    else:
                        Colors.error(f"pca_scatter_inject.html not found in: {_inject_base} — please try again.")
                        continue
                else:
                    Colors.error("Path not found — please try again.")
                    continue

                Colors.success(f"Inject file found: {_inject_path}")
                _use_interactive = True
                break

            # ── HTML symbol file ──────────────────────────────────────────
            _html_sym_raw = input(
                f"{Colors.CYAN}  Enter full path or folder containing "
                f"symbol_conversion_html.txt\n"
                f"  (HTML symbols for hover labels, or press [Enter] to reuse "
                f"main symbol file): {Colors.ENDC}"
            ).strip().strip('"\'')

            if _html_sym_raw:
                _html_sym_base = os.path.abspath(os.path.expanduser(_html_sym_raw))

                if os.path.isfile(_html_sym_base):
                    _sym_html = SymbolConverter(_html_sym_base)
                    Colors.success(f"HTML symbol file loaded: {_html_sym_base}")
                elif os.path.isdir(_html_sym_base):
                    _candidate = os.path.join(_html_sym_base, 'symbol_conversion_html.txt')
                    if os.path.isfile(_candidate):
                        _sym_html = SymbolConverter(_candidate)
                        Colors.success(f"HTML symbol file found (auto-detected): {_candidate}")
                    else:
                        Colors.warning(
                            f"symbol_conversion_html.txt not found in: {_html_sym_base} "
                            f"— falling back to main symbol file."
                        )
                        _sym_html = sym
                else:
                    Colors.warning("Path not found — falling back to main symbol file.")
                    _sym_html = sym
            else:
                Colors.info("Reusing main symbol file for hover labels.")
                _sym_html = sym

    # ── Load data ──────────────────────────────────────────────────────────
    _t0 = time.perf_counter()
    df = DataLoader.load_folder(data_folder, file_subset=_file_subset, n_cpus=n_cpus, backend_cfg=backend_cfg)
    _timing_log['Data loading'] = time.perf_counter() - _t0
    print(f"  Data loading: {time.perf_counter()-_t0:.3f} s")
    # Restrict to selected features only (drop any that didn't survive loading)
    feature_cols = [c for c in feature_cols if c in df.columns]
    sym.report_coverage(feature_cols)

    # ── PCA ────────────────────────────────────────────────────────────────
    runner           = PCARunner(n_components=5)
    df_pca, loadings = runner.fit_transform(df, feature_cols)
    print_category_summary(df_pca)

    dt_col = next((c for c in ['DT', 'DT_AGL', 'DT_A_AGL'] if c in df_pca.columns), 'DT')   
    
    # ─────────────────────────────────────────────────────────────────────────
    # BLOCK 1
    # ─────────────────────────────────────────────────────────────────────────
    _t_block = time.perf_counter()
    Colors.block(1, "LATENT DESCRIPTOR SPACE CONSTRUCTION VIA EIGENVECTOR DECOMPOSITION")
    feature_labels = ask_feature_labels(feature_cols, sym)

    Colors.info("[1.1] Generating Explained Variance Bar Chart")
    _t0 = time.perf_counter()
    plot_explained_variance(runner.explained_variance, save_base)
    print(f"  plot_explained_variance: {time.perf_counter()-_t0:.3f} s")
    Colors.success("BLOCK 1 COMPLETE")
    _timing_log['Block 1 — Explained variance'] = time.perf_counter() - _t0
    print(f"  Block 1 total: {time.perf_counter()-_t_block:.3f} s")

    # ─────────────────────────────────────────────────────────────────────────
    # BLOCK 2
    # ─────────────────────────────────────────────────────────────────────────
    _t_block = time.perf_counter()
    Colors.block(2, "LATENT SPACE MAPPING & CLUSTERING ANALYSIS")
    show_labels = False  # formula labels disabled; uncomment ask_label_option() below to re-enable
    # show_labels = ask_label_option()

    Colors.info("[2.1] Generating PC Loading Biplot")
    _t0 = time.perf_counter()
    plot_loadings(loadings, feature_labels, save_base)
    print(f"  plot_loadings: {time.perf_counter()-_t0:.3f} s")
    
    Colors.info("[2.2] PC1 vs PC2 Scatter Plot")
    _t0 = time.perf_counter()
    plot_scatter(df_pca, save_base, show_labels)
    if _use_interactive and _inject_path:
        plot_scatter_interactive(df_pca, feature_cols, _sym_html, save_base, runner,
                                 inject_path=_inject_path)
    elif not PLOTLY_AVAILABLE:
        Colors.warning("Skipping interactive scatter — plotly not installed.")
    print(f"  plot_scatter: {time.perf_counter()-_t0:.3f} s")
    Colors.success("BLOCK 2 COMPLETE")
    _timing_log['Block 2 — Scatter plots'] = time.perf_counter() - _t0
    print(f"  Block 2 total: {time.perf_counter()-_t_block:.3f} s")

    # ─────────────────────────────────────────────────────────────────────────
    # BLOCK 3
    # ─────────────────────────────────────────────────────────────────────────
    _t_block = time.perf_counter()
    Colors.block(3, "PROPERTY FIELD PROJECTION IN REDUCED PCA SPACE")

    Colors.info("[3.1] Material Count vs PC1/PC2")
    _t0 = time.perf_counter()
    plot_material_count_vs_components(df_pca, save_base)
    _t_count = time.perf_counter() - _t0
    print(f"  plot_material_count_vs_components: {_t_count:.3f} s")

    Colors.info("[3.2] Θ_D Statistics vs PC1/PC2")
    _t0 = time.perf_counter()
    plot_theta_d_statistics_vs_components(df_pca, save_base, sym, dt_col)
    _t_stats = time.perf_counter() - _t0
    print(f"  plot_theta_d_statistics_vs_components: {_t_stats:.3f} s")

    Colors.info("[3.3] Material Count vs Θ_D Distribution")
    _t0 = time.perf_counter()
    plot_theta_d_distribution(df_pca, save_base, sym, dt_col)
    _t_dist = time.perf_counter() - _t0
    print(f"  plot_theta_d_distribution: {_t_dist:.3f} s")

    Colors.info("[3.4] Property Contour Maps")
    _t0 = time.perf_counter()
    plot_contours(df_pca, feature_cols, save_base, sym)
    _t_contour = time.perf_counter() - _t0
    print(f"  plot_contours: {_t_contour:.3f} s")

    Colors.success("BLOCK 3 COMPLETE")
    _timing_log['Block 3 — Count vs PC1/PC2']            = _t_count
    _timing_log['Block 3 — ThetaD stats vs PC']          = _t_stats
    _timing_log['Block 3 — ThetaD distribution']         = _t_dist
    _timing_log['Block 3 — Contour maps']                = _t_contour
    _timing_log['Block 3 — Property projections (wall)'] = time.perf_counter() - _t_block
    print(f"  Block 3 total: {time.perf_counter()-_t_block:.3f} s")
    # ─────────────────────────────────────────────────────────────────────────
    # BLOCK 4
    # ─────────────────────────────────────────────────────────────────────────
    _t_block = time.perf_counter()
    Colors.block(4, "THRESHOLD-BASED BIFURCATION & SCREENING")
    Colors.info("[4.1] Property Threshold Analysis")
    _t0 = time.perf_counter()
    plot_property_threshold_analysis(df_pca, save_base, sym, dt_col)
    print(f"  plot_property_threshold_analysis: {time.perf_counter()-_t0:.3f} s")
    Colors.success("BLOCK 4 COMPLETE")
    _timing_log['Block 4 — Threshold analysis'] = time.perf_counter() - _t0
    print(f"  Block 4 total: {time.perf_counter()-_t_block:.3f} s")

    # ─────────────────────────────────────────────────────────────────────────
    # EXCEL EXPORT 
    # ─────────────────────────────────────────────────────────────────────────
    _timing_log['TOTAL wall-clock'] = time.perf_counter() - _t_pipeline  
    _t0 = time.perf_counter()
    Colors.block("E", "EXCEL DATA EXPORT")
    xl_path = export_to_excel(
        df_pca           = df_pca,
        loadings         = loadings,
        runner           = runner,
        feature_cols     = feature_cols,
        symbol_converter = sym,
        save_base        = save_base,
        dt_col           = dt_col,
        timing_log       = _timing_log,
        n_cpus           = n_cpus,                  # ← add
        backend_label    = backend_cfg['label'],    # ← add
    )
    src_path = export_figure_source_data(
        df_pca=df_pca, loadings=loadings, runner=runner,
        feature_cols=feature_cols, symbol_converter=sym,
        save_base=save_base, dt_col=dt_col,
    )
    _timing_log['Excel export'] = time.perf_counter() - _t0
    print(f" EXCEL EXPORT: {time.perf_counter()-_t0:.3f} s")

    # ── Final summary ──────────────────────────────────────────────────────
    Colors.header("ALL ANALYSES COMPLETE")
    Colors.success(f"All PDF figures saved to: {os.path.dirname(save_base)}")
    Colors.success(f"Excel workbook saved to:  {xl_path}")
    Colors.success(f"Figure source data saved to: {src_path}")
    print(f"\n{Colors.CYAN}Analysis Summary:{Colors.ENDC}")
    print(f"  Total materials analysed : {len(df_pca):,}")
    print(f"  PDF figures generated    : 8 PDFs")
    print(f"  Excel sheets exported    : 13 sheets")
    print(f"  Output directory         : {os.path.dirname(save_base)}\n")
    print(f"\n  Total wall-clock time: {_timing_log['TOTAL wall-clock']:.3f} s")


if __name__ == "__main__":
    main()


                       PCA ANALYSIS OF MATERIALS DATASET                        

  ℹ  Environment: Jupyter | Local | vCPUs available: 16

  vCPUs available: 16

  How many vCPU (virtual cores) to use? (max: 16)
  Press [Enter] to use 1 (safe default), or enter a number: 

 16


  ✓  Using 16 vCPU(s)  |  Backend: joblib threads (16 cores) — Jupyter safe

[STEP 1] SELECT DATA FOLDER
--------------------------------------------------------------------------------



  Enter full location of folder path containing classified Excel files:  C:/classified_data/


  ✓  Folder found: C:/classified_data/

[STEP 2] SELECT FILES & FEATURES
--------------------------------------------------------------------------------

  Files found:
    [1]  Boride.xlsx
    [2]  Carbide.xlsx
    [3]  Element.xlsx
    [4]  Halide.xlsx
    [5]  Metal-Metalloid.xlsx
    [6]  Nitride.xlsx
    [7]  Other.xlsx
    [8]  Oxide.xlsx


...

  ✓  Symbol file found (auto-detected): C:\symbol_conversion.txt
  ✓  Loaded 38 symbol mappings from C:\symbol_conversion.txt

[STEP 5] INTERACTIVE SCATTER PLOT
--------------------------------------------------------------------------------



  Generate interactive HTML scatter plot? (y/n) [Default: n]:  y
  Enter full path or folder containing pca_scatter_inject.html:  C:/


  ✓  Inject file found: C:\pca_scatter_inject.html


  Enter full path or folder containing symbol_conversion_html.txt
  (HTML symbols for hover labels, or press [Enter] to reuse main symbol file):  C:/


  ✓  Loaded 38 symbol mappings from C:\symbol_conversion_html.txt
  ✓  HTML symbol file found (auto-detected): C:\symbol_conversion_html.txt

[STEP 6] LOADING DATA
--------------------------------------------------------------------------------
  ℹ  Loading with: joblib threads (16 cores) — Jupyter safe
  ✓  Boride.xlsx  →  'Borides'  (412 rows)
  ✓  Carbide.xlsx  →  'Carbides'  (462 rows)
  ✓  Element.xlsx  →  'Single Element'  (294 rows)
  ✓  Halide.xlsx  →  'Halides'  (683 rows)
  ✓  Metal-Metalloid.xlsx  →  'Metal & Metalloid Compounds'  (6,582 rows)
  ✓  Nitride.xlsx  →  'Nitrides'  (535 rows)
  ✓  Other.xlsx  →  'Other Compounds'  (1,413 rows)
  ✓  Oxide.xlsx  →  'Oxides'  (1,128 rows)
  ℹ  Total rows loaded: 11,509
  Data loading: 3.206 s
  ✓  Symbol mappings found for: ['SM', 'YM', 'VPA', 'D', 'DT']

BLOCK 1: LATENT DESCRIPTOR SPACE CONSTRUCTION VIA EIGENVECTOR DECOMPOSITION

  Scaler: 0.005 s
  PCA decomposition: 0.012 s

  Component      Variance   Cumulative
  --------------